# CADPS2 - Single Gene Analysis in GP2 NBA data

In [1]:
analysis = "NBA"
# analysis = "WGS"

## Description

1. Description

- Getting Started
- Loading Python libraries
- Defining functions 
- Set Paths 
- Make working directory 

2. Installing packages

3. Copy over data

4. Create a covariate file with GP2 data

5. Annotation of the gene

6. Case/Control Frequencies

5. Calculate frequency in cases versus controls

6. Calculate frequency (homozygotes) in cases versus controls

7. Save out results

## Loading Python libraries

In [2]:
# Use the os package to interact with the environment
import os

#Import Sys
import sys as sys

# Use pathlib for file path manipulation
import pathlib
from pathlib import Path

# Parallel processing
from itertools import product
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

# Import subprocess for easier management of bash commands
import subprocess

# Bring in Pandas for Dataframe functionality
import pandas as pd

# Numpy for basics
import numpy as np

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib
import seaborn as sns

# Import date and time for timestamps
from datetime import datetime, date
import time


print("Date of first run: 06-15-2025")
print("Date of last edits: 07-30-2025")


### Last iteration
print("Last iteration at:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

### Package versions

print(f'''
=========================
    PACKAGE VERSIONS
=========================
pandas: {pd.__version__}
numpy: {np.__version__}
matplotlib: {matplotlib.__version__}
seaborn: {sns.__version__}
''')


Date of first run: 06-15-2025
Date of last edits: 07-30-2025
Last iteration at: 2026-03-02 14:38:17

    PACKAGE VERSIONS
pandas: 2.2.1
numpy: 1.26.0
matplotlib: 3.10.1
seaborn: 0.13.2



## Set paths

### Release paths

In [3]:
## GP2 v11
REL11_PATH = pathlib.Path(pathlib.Path.home(), 'workspace/gp2_tier2_eu_release11')
!ls -hal {REL11_PATH}

total 21K
dr-xr-xr-x. 1 jupyter users   0 Mar  2 12:56 clinical_data
dr-xr-xr-x. 1 jupyter users   0 Mar  2 12:56 clinical_exomes
dr-xr-xr-x. 1 jupyter users   0 Mar  2 12:56 imputed_genotypes
dr-xr-xr-x. 1 jupyter users   0 Mar  2 12:56 meta_data
dr-xr-xr-x. 1 jupyter users   0 Mar  2 12:56 raw_genotypes
dr-xr-xr-x. 1 jupyter users   0 Mar  2 12:56 raw_genotypes_flipped
-r--r--r--. 1 jupyter users 21K Dec  2 04:24 README_R11.txt
dr-xr-xr-x. 1 jupyter users   0 Mar  2 12:56 wgs


In [4]:
# GP2 R11

CLINICAL_DATA_PATH = pathlib.Path(REL11_PATH, 'clinical_data/master_key_release11_final_vwb.csv')
EXTENDED_CLINICAL_DATA_PATH = pathlib.Path('REL11_PATH, clinical_data/r11_extended_clinical_data_vwb.csv')

IMPUTED_GENO_PATH = pathlib.Path(REL11_PATH, 'imputed_genotypes')
RAW_GENO_PATH = pathlib.Path(REL11_PATH, 'raw_genotypes')

RELATED_DATA_PATH = pathlib.Path(REL11_PATH, 'meta_data/related_samples')

In [5]:
## Use tree command to visualize directory structure
## To see more depth, adjust -L flag (e.g. 1, 2, 3)
! tree -L 1 {REL11_PATH}

/home/jupyter/workspace/gp2_tier2_eu_release11
├── clinical_data
├── clinical_exomes
├── imputed_genotypes
├── meta_data
├── raw_genotypes
├── raw_genotypes_flipped
├── README_R11.txt
└── wgs

7 directories, 1 file


## Workspace paths

In [ ]:
GENE_DATA = """
CADPS2 7 122318311 122886859
"""

In [7]:
GENE_DICT = {}
for line in GENE_DATA.strip().split("\n"):
    name, chrom, start, end = line.split()
    GENE_DICT[name] = {
        "CHR": chrom,
        "START": int(start),
        "END": int(end)
    }


In [8]:
print(GENE_DICT)

{'CADPS2': {'CHR': '7', 'START': 122318311, 'END': 122886859}}


In [9]:
## Define gene of interest
# GENE_NAME = "CADPS2"

for GENE in GENE_DICT:
    print(GENE)


RELEASE = "NBA_R11" 

CADPS2


In [10]:
### Define ANCESTRIES as a python list
ANCESTRIES = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']

In [11]:
## Get main directories

GENE_NAME = "CADPS2"

TOP_DIR =   "/home/jupyter"
TOOL_DIR =  "/home/jupyter/tools"
HOME_DIR =  "/home/jupyter/workspace"
WS_DIR=     "/home/jupyter/workspace/ws_files/"
WORK_DIR =  "/home/jupyter/workspace/ws_files/NahuTestR11"
ANA_DIR =   f"/home/jupyter/workspace/ws_files/NahuTestR11/{analysis}"
RESU_DIR =  "/home/jupyter/workspace/ws_files/NahuTestR11/Results"

dirs = ("/home/jupyter/tools", "/home/jupyter/workspace/ws_files/NahuTestR11", f"/home/jupyter/workspace/ws_files/NahuTestR11/{analysis}")

## Subdirs for analyses
subdirs = ["InputFiles", "VariantDescriptives", "Association", "GLM", "Burden"]

for dir in dirs:
    os.makedirs(dir, exist_ok=True)       
    os.makedirs(f"{RESU_DIR}/{analysis}", exist_ok = True)         ## exist_ok = True checks if already created
for ANCESTRY in ANCESTRIES:
    for subdir in subdirs:
        os.makedirs(f"{ANA_DIR}/{ANCESTRY}/{subdir}", exist_ok = True) 

## Install Packages

In [ ]:
%%capture
%%bash

# Install plink 1.9

## Change directory to tools
mkdir $TOOL_DIR
cd $TOOL_DIR

## ------------------------------## 
## Check if plink1.9 is installed
if test -e /home/jupyter/tools/plink1.9; then
    echo "Plink is already installed in /home/jupyter/tools"
else
    echo "Plink is not installed"

    cd /home/jupyter/tools

    wget -nc http://s3.amazonaws.com/plink1-assets/plink_linux_x86_64_20190304.zip 

    unzip -o plink_linux_x86_64_20190304.zip
    mv plink plink1.9
fi

## ------------------------------## 
## Check if plink2.0 is installed
if test -e /home/jupyter/tools/plink2; then
    echo "Plink2 is already installed in /home/jupyter/tools"
else
    echo "Plink2 is not installed"
    cd /home/jupyter/tools
    
    wget -nc http://s3.amazonaws.com/plink2-assets/plink2_linux_x86_64_latest.zip

    unzip -o plink2_linux_x86_64_latest.zip
fi

In [ ]:
# chmod (change permissions): plink1.9 and plink2 to make sure you have permission to run the program
! chmod u+x /home/jupyter/tools/plink1.9
! chmod u+x /home/jupyter/tools/plink2

In [ ]:
%%capture
%%bash

# Install ANNOVAR: We are adding the download link after registration on the annovar website
# https://www.openbioinformatics.org/annovar/annovar_download_form.php

if test -e /home/jupyter/tools/annovar; then
    echo "annovar is already installed in /home/jupyter/tools/notebooks"

else
    echo "annovar is not installed"
    cd /home/jupyter/tools
    
    wget -nc http://www.openbioinformatics.org/annovar/download/0wgxR2rIVP/annovar.latest.tar.gz
    
    tar -xvzf annovar.latest.tar.gz
fi

In [ ]:
%%capture
%%bash

cd /home/jupyter/tools/annovar/

# Install ANNOVAR: Download resources for annotation

#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar dbnsfp47a humandb/ #latest version of dbNSFP
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar clinvar_20250721 humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad41_genome humandb/
#perl annotate_variation.pl -buildver hg38 -downdb -webfrom annovar gnomad41_exome humandb/

In [ ]:
%%capture
%%bash

#Install RVTESTS: Option 1 (~15min)
if test -e /home/jupyter/tools/rvtests; then
    echo "RVtests is already installed in /home/jupyter/tools/"
else
    echo "RVtests is not installed"
    
    mkdir /home/jupyter/tools/rvtests
    cd /home/jupyter/tools/rvtests
    
    wget https://github.com/zhanxw/rvtests/releases/download/v2.1.0/rvtests_linux64.tar.gz 
    
    tar -zxvf rvtests_linux64.tar.gz
fi

In [ ]:
! chmod 777 /home/jupyter/tools/rvtests/executable/rvtest

In [ ]:
! {TOOL_DIR}/rvtests/executable/rvtest --help

In [ ]:
## Download refFlat

! wget -O {TOOL_DIR}/refFlat_HG38_all_chr.txt.gz https://hgdownload.soe.ucsc.edu/goldenPath/hg38/database/refFlat.txt.gz 
! gunzip -f {TOOL_DIR}/refFlat_HG38_all_chr.txt.gz

In [ ]:
'''%%bash
cd /home/jupyter/tools/
rm *.zip
rm *.gz
rm toy*
rm prettify
rm vcf_subset
rm LICENSE
rm ./rvtests/*.gz
rm -r  ./rvtests/example'''

## Create a covariate file with GP2 data

In [83]:
# Let's load the master key
key1 = pd.read_csv(CLINICAL_DATA_PATH, low_memory=False)
print(f'Clinical data (num rows, num columns): {key1.shape}')
pd.set_option('display.max_columns', None)
key1.head()

Clinical data (num rows, num columns): (130274, 32)


,study,FID,GP2ID,nba,wgs,clinical_exome,extended_clinical_data,nba_prune_reason,nba_related,nba_label,wgs_prune_reason,wgs_label,study_arm,study_type,diagnosis,baseline_GP2_phenotype_for_qc,baseline_GP2_phenotype,biological_sex_for_qc,age_at_sample_collection,age_of_onset,age_at_diagnosis,age_at_death,age_at_last_follow_up,race_for_qc,family_history_for_qc,region_for_qc,manifest_id,genotyping_site,sample_type,amppd_wgs,GDPR,flag
0,24HR,24HR_000001,24HR_000001,1.0,1.0,NaN,NaN,NaN,NaN,EUR,NaN,EUR,Case,Case(/Control),PD,PD,PD,Female,NaN,NaN,NaN,NaN,NaN,Not Reported,Not Reported,USA,m1,Psomagen,DNA,NaN,0.0,NaN
1,24HR,24HR_000002,24HR_000002,1.0,1.0,NaN,NaN,NaN,NaN,EUR,NaN,EUR,Case,Case(/Control),PD,PD,PD,Male,NaN,NaN,NaN,NaN,NaN,Not Reported,Not Reported,USA,m1,Psomagen,DNA,NaN,0.0,NaN
2,24HR,24HR_000003,24HR_000003,1.0,1.0,NaN,NaN,NaN,NaN,EUR,NaN,EUR,Case,Case(/Control),PD,PD,PD,Male,NaN,NaN,NaN,NaN,NaN,Not Reported,Not Reported,USA,m1,Psomagen,DNA,NaN,0.0,NaN
3,24HR,24HR_000004,24HR_000004,1.0,1.0,NaN,NaN,NaN,NaN,EUR,NaN,EUR,Case,Case(/Control),PD,PD,PD,Male,NaN,NaN,NaN,NaN,NaN,Not Reported,Not Reported,USA,m1,Psomagen,DNA,NaN,0.0,NaN
4,24HR,24HR_000005,24HR_000005,1.0,1.0,NaN,NaN,NaN,NaN,EUR,NaN,EUR,Case,Case(/Control),PD,PD,PD,Male,NaN,NaN,NaN,NaN,NaN,Not Reported,Not Reported,USA,m1,Psomagen,DNA,NaN,0.0,NaN


In [84]:
# Subsetting to keep only a few columns 
key = key1[['GP2ID', 'GP2ID', 'baseline_GP2_phenotype_for_qc', 'biological_sex_for_qc', 'age_at_sample_collection', 'age_of_onset', 'nba_label','wgs_label']]

# 'nba_GP2sampleID_r11'
# Renaming the columns

key.columns = ['IID', 'IID1_OLD', 'phenotype', 'SEX', 'AGE', 'AAO', 'nba_label', 'wgs_label']
key

,IID,IID1_OLD,phenotype,SEX,AGE,AAO,nba_label,wgs_label
0,24HR_000001,24HR_000001,PD,Female,NaN,NaN,EUR,EUR
1,24HR_000002,24HR_000002,PD,Male,NaN,NaN,EUR,EUR
2,24HR_000003,24HR_000003,PD,Male,NaN,NaN,EUR,EUR
3,24HR_000004,24HR_000004,PD,Male,NaN,NaN,EUR,EUR
4,24HR_000005,24HR_000005,PD,Male,NaN,NaN,EUR,EUR
...,...,...,...,...,...,...,...,...
130269,YMS_000080,YMS_000080,Other,Male,66.0,NaN,AJ,NaN
130270,YMS_000081,YMS_000081,Other,Male,71.0,NaN,AAC,NaN
130271,YMS_000082,YMS_000082,PD,Male,76.0,71.0,AJ,NaN
130272,YMS_000083,YMS_000083,Other,Male,69.0,NaN,AJ,NaN


In [85]:
key["label"] = key["nba_label"].combine_first(key["wgs_label"])
key = key.drop(columns=["nba_label", "wgs_label"])
key

/tmp/ipykernel_658/428804967.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  key["label"] = key["nba_label"].combine_first(key["wgs_label"])


,IID,IID1_OLD,phenotype,SEX,AGE,AAO,label
0,24HR_000001,24HR_000001,PD,Female,NaN,NaN,EUR
1,24HR_000002,24HR_000002,PD,Male,NaN,NaN,EUR
2,24HR_000003,24HR_000003,PD,Male,NaN,NaN,EUR
3,24HR_000004,24HR_000004,PD,Male,NaN,NaN,EUR
4,24HR_000005,24HR_000005,PD,Male,NaN,NaN,EUR
...,...,...,...,...,...,...,...
130269,YMS_000080,YMS_000080,Other,Male,66.0,NaN,AJ
130270,YMS_000081,YMS_000081,Other,Male,71.0,NaN,AAC
130271,YMS_000082,YMS_000082,PD,Male,76.0,71.0,AJ
130272,YMS_000083,YMS_000083,Other,Male,69.0,NaN,AJ


In [86]:
for ANCESTRY in ANCESTRIES:
        ## To run analyses here
        
        print(f'\nWORKING ON: {ANCESTRY}')
            
        ## Subset to keep ANCESTRY of interest 
        ANCESTRY_key = key[key['label']==ANCESTRY].copy()
        ANCESTRY_key.reset_index(drop=True)
            
        # Load information about related individuals in the ANCESTRY analyzed
        related_df = pd.read_csv(f'{RELATED_DATA_PATH}/{ANCESTRY}_release11_vwb.related')
        print(f'Related individuals: {related_df.shape}')
        
        # Make a list of just one set of related people
        related_list = list(related_df['IID1'])
        
        # Check value counts of related and remove only one related individual
        ANCESTRY_key = ANCESTRY_key[~ANCESTRY_key["IID1_OLD"].isin(related_list)]
        # Check size
        print(f'Unrelated individuals: {ANCESTRY_key.shape}')
            
        # Convert phenotype to binary (1/2)
        ## Assign conditions so case=2 and controls=1, and -9 otherwise (matching PLINK convention)
        # PD = 2; control = 1
        pheno_mapping = {"PD": 2, "Control": 1}
        ANCESTRY_key['PHENO'] = ANCESTRY_key['phenotype'].map(pheno_mapping).astype('Int64')
    
        # Check value counts of pheno
        ANCESTRY_key['PHENO'].value_counts(dropna=False)
        
        ## Get the PCs
        pcs = pd.read_csv(f'{RAW_GENO_PATH}/{ANCESTRY}/{ANCESTRY}_release11_vwb.eigenvec', sep='\t')
        
        #Select just first 10 PCs
        selected_columns = ['FID', 'IID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']
        pcs = pd.DataFrame(data=pcs.iloc[:, 0:12].values, columns=selected_columns)
    
        # Drop the first row (since it's now the column names)
        pcs = pcs.drop(0)
    
        # Reset the index to remove any potential issues
        pcs = pcs.reset_index(drop=True)
        
        # Check size
        print(f'PCs: {pcs.shape}')
    
        # Check value counts of SEX
        sex_og_values = ANCESTRY_key['SEX'].value_counts(dropna=False)
        print(f'Sex value counts - original:\n {sex_og_values.to_string()}')
        
        # Convert sex to binary (1/2)
        ## Assign conditions so female=2 and men=1, and -9 otherwise (matching PLINK convention)
        # Female = 2; Male = 1
        sex_mapping = {"Female": 2, "Male": 1}
        ANCESTRY_key['SEX'] = ANCESTRY_key['SEX'].map(sex_mapping).astype('Int64')
        
        # Check value counts of SEX after recoding
        sex_recode_values = ANCESTRY_key['SEX'].value_counts(dropna=False)
        print(f'Sex value counts - recoded:\n{sex_recode_values.to_string()}')
        
        ## Make covariate file
        df = pd.merge(pcs, ANCESTRY_key, on='IID', how='left')
        print(f'\nCheck columns for covariate file: {df.columns}')
        
        #Make additional columns - FID, fatid and matid - these are needed for RVtests!!
        #RVtests needs the first 5 columns to be fid, iid, fatid, matid and sex otherwise it does not run correctly
        #Uppercase column name is ok
        #See https://zhanxw.github.io/rvtests/#phenotype-file
        df['FATID'] = 0
        df['MATID'] = 0
    
        ## Clean up and keep columns we need 
        final_df = df[['FID', 'IID', 'FATID', 'MATID', 'SEX', 'AGE', 'PHENO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']].copy()
        final_df.columns = ['FID', 'IID', 'FATID', 'MATID', 'SEX', 'AGE', 'PHENO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']
    
        print(f'\nCheck columns for Final covariate file: {final_df.columns}')
    
        ##DO NOT replace missing values with -9 as this is misinterpreted by RVtests - needs to be nonnumeric
        #Leave missing values as NA
        
        #Check number of PD cases missing age
        pd_missAge = final_df[(final_df['PHENO']==2)&(final_df['AGE'].isna())]
        print(f'Number of PD cases missing age: {pd_missAge.shape[0]}')
        
        #Check number of controls missing age
        control_missAge = final_df[(final_df['PHENO']==1)&(final_df['AGE'].isna())]
        print(f'Number of controls missing age: {control_missAge.shape[0]}')
    
        ## Make file of sample IDs to keep 
        samples_toKeep = final_df[['FID', 'IID']].copy()
        
        samplestokeep_path = pathlib.Path(pathlib.Path.home(), f'{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}.samplestokeep')
        
        # Create the output CSV file's parent folder in the cloud storage bucket, if it doesn't already exist.
        if not samplestokeep_path.parent.exists():
            !mkdir -p {samplestokeep_path.parent}
            print(f'Created {samplestokeep_path.parent}')
        
        samples_toKeep.to_csv(samplestokeep_path, sep = '\t', index=False, header=None)
    
        finaldf_path = pathlib.Path(pathlib.Path.home(), f'{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_covariate_file.txt')
        
        # Create the output CSV file's parent folder in the cloud storage bucket, if it doesn't already exist.
        if not finaldf_path.parent.exists():
            !mkdir -p {finaldf_path.parent}
            print(f'Created {finaldf_path.parent}')
        
        final_df.to_csv(finaldf_path, sep = '\t', na_rep='NA', index=False)


WORKING ON: AAC
Related individuals: (12, 9)
Unrelated individuals: (1584, 7)
PCs: (1381, 12)
Sex value counts - original:
 SEX
Female                        861
Male                          721
Other/Unknown/Not Reported      2
Sex value counts - recoded:
SEX
2       861
1       721
<NA>      2

Check columns for covariate file: Index(['FID', 'IID', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8',
       'PC9', 'PC10', 'IID1_OLD', 'phenotype', 'SEX', 'AGE', 'AAO', 'label',
       'PHENO'],
      dtype='object')

Check columns for Final covariate file: Index(['FID', 'IID', 'FATID', 'MATID', 'SEX', 'AGE', 'PHENO', 'PC1', 'PC2',
       'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10'],
      dtype='object')
Number of PD cases missing age: 82
Number of controls missing age: 22

WORKING ON: AFR
Related individuals: (429, 9)
Unrelated individuals: (7937, 7)
PCs: (7245, 12)
Sex value counts - original:
 SEX
Male                          4658
Female                        3277

## Annotation of the gene

### Extract the region using PLINK in hg38

- Extract *CADPS2* gene
- *CADPS2* coordinates: Chromosome 7: 122,318,311-122,886,859 (GRCh38/hg38)
- Source: https://www.genecards.org/cgi-bin/carddisp.pl?gene=CADPS2

In [87]:
## extract region using plink

def regionExtractor(number, startBP, endBP, ANCESTRY):
    
    ## Analysis start time
    ts_start = datetime.now().strftime('Date: %Y-%m-%d   Time: %H-%M-%S')
    print(f"\n\n=============   ANALYSIS START:   {ts_start}   =====================\n")
    
    ## Get ANCESTRY name
    print(f"\nExtracting {GENE_NAME} from {ANCESTRY} NBA Release 11 data.")
    
    ## Create directory for ANCESTRY
    print(f"\nStoring output plink2 files in {ANA_DIR}/{ANCESTRY}\n\n")

    inputRawFile = f"{IMPUTED_GENO_PATH}/{ANCESTRY}/chr{number}_{ANCESTRY}_release11_vwb"
    outputPrefix = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_{GENE_NAME}"
    
    ## Define plink2 command for extraction
    regionExtractor = [
        f"{TOOL_DIR}/plink2",
        "--pfile", inputRawFile,
        "--chr", str(number),
        "--from-bp", str(startBP),
        "--to-bp", str(endBP),
        "--extract-if-info", "R2>=0.8",
        "--mac", "2",
        "--hwe", "0.0001", 'keep-fewhet',
        "--max-maf", "0.05",
        "--make-pgen",
        "--out", outputPrefix
    ]

    ## Run plink2 command as bash subprocess
    result = subprocess.run(regionExtractor, capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)

    
    vcfConverter = [
        f"{TOOL_DIR}/plink2",
        "--pfile", outputPrefix,
        "--recode", "vcf", "id-paste=iid",
        "--out", outputPrefix     
    ]

    ## Run plink2 command as bash subprocess
    subprocess.run(vcfConverter, check=True)
    
    vcfGzip = [
        "bgzip",
        "-f", f"{outputPrefix}.vcf"
    ]

    ## Run plink2 command as bash subprocess
    subprocess.run(vcfGzip, check=True)
    
    vcfIndex = [
        "tabix",
        "-f", "-p", "vcf", f"{outputPrefix}.vcf.gz"     
    ]    
    ## Run plink2 command as bash subprocess
    subprocess.run(vcfIndex, check=True)
    
    
    ## Analysis end time
    ts_finish = datetime.now().strftime('Date: %Y-%m-%d   Time: %H-%M-%S')
    print(f"\n\n=============   ANALYSIS END:   {ts_finish}   =====================\n")
    
    ## Return ANCESTRY, position and gene
    return f"Extracted: {ANCESTRY}, chr{number}:{startBP}:{endBP} for {GENE_NAME}"

In [88]:
for GENE_NAME in GENE_DICT:
    max_workers = max(1, len(ANCESTRIES) // 2)
    BUILD = "hg38"
    chrom = GENE_DICT[GENE_NAME]["CHR"]
    startBP = str(GENE_DICT[GENE_NAME]["START"])
    endBP = str(GENE_DICT[GENE_NAME]["END"])
    number = chrom.replace("chr", "")  # remove "chr" if needed

    print(f"""\nRegion of interest: chr{number}:{startBP}:{endBP} (build version {BUILD}), {GENE_NAME}\nANCESTRIES to extract: {ANCESTRIES}""")

    for ANCESTRY in ANCESTRIES:
        print(f'\nWORKING ON: {ANCESTRY}\n')
        BUILD = "hg38"
        
        regionExtractor(7, 122318311, 122886859, f"{ANCESTRY}")


Region of interest: chr7:122318311:122886859 (build version hg38), CADPS2
ANCESTRIES to extract: ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']

WORKING ON: AAC



=============   ANALYSIS START:   Date: 2026-02-25   Time: 19-28-17   =====================


Extracting CADPS2 from AAC NBA Release 11 data.

Storing output plink2 files in /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AAC


PLINK v2.0.0-a.7LM 64-bit Intel (18 Nov 2025)       cog-genomics.org/plink/2.0/
(C) 2005-2025 Shaun Purcell, Christopher Chang    GNU General Public License v3
Logging to /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AAC/InputFiles/AAC_CADPS2.log.
Options in effect:
  --chr 7
  --extract-if-info R2>=0.8
  --from-bp 122318311
  --hwe 0.0001 keep-fewhet
  --mac 2
  --make-pgen
  --max-maf 0.05
  --out /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AAC/InputFiles/AAC_CADPS2
  --pfile /home/jupyter/workspace/gp2_tier2_eu_release11/imputed_genotypes/AAC/chr7_AAC_release

## Annotate using ANNOVAR

In [15]:
def regionAnnotator(GENE_NAME, ANCESTRY):
    print(f"Extracting {GENE_NAME} from {ANCESTRY} NBA Release 11 data.")
    
    input_vcf = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_{GENE_NAME}.vcf.gz"
    output_base = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_{GENE_NAME}.annovar"
    
    cmd = [
        "perl", f"{TOOL_DIR}/annovar/table_annovar.pl",
        input_vcf,
        f"{TOOL_DIR}/annovar/humandb/",
        "-buildver", "hg38",
        "-out", output_base,
        "-remove", 
        "-protocol", "refGene,clinvar_20250721,dbnsfp47a,gnomad41_genome,gnomad41_exome", 
        "-operation", "g,f,f,f,f", 
        "--nopolish",
        "-nastring", ".",
        "-vcfinput"
    ]
    
    try:
        subprocess.run(cmd, check=True)
        print(f"\nSUCCESS: {ANCESTRY}, {GENE_NAME}\n")
    except subprocess.CalledProcessError as e:
        print(f"\nERROR in {ANCESTRY}, {GENE_NAME}: {e}\n")
    
    return f"{output_base}.hg38_multianno.txt" # Return the path so you can use it

# Logic for workers and product
def regionAnnotator_wrapper(args):
    return regionAnnotator(*args)

# Explicitly use .keys() if GENE_DICT is a dictionary
args = list(product(GENE_DICT.keys(), ANCESTRIES)) 

max_workers = max(1, len(ANCESTRIES) // 2)

with ProcessPoolExecutor(max_workers=max_workers) as executor:
    # results will now be a list of paths to your output files
    results = list(executor.map(regionAnnotator_wrapper, args))

Extracting CADPS2 from AFR NBA Release 11 data.Extracting CADPS2 from AJ NBA Release 11 data.Extracting CADPS2 from CAS NBA Release 11 data.Extracting CADPS2 from AMR NBA Release 11 data.Extracting CADPS2 from AAC NBA Release 11 data.







NOTICE: Running with system command <convert2annovar.pl  -includeinfo -allsample -withfreq -format vcf4 /home/jupyter/workspace/ws_files/NahuTestR11/NBA/CAS/InputFiles/CAS_CADPS2.vcf.gz > /home/jupyter/workspace/ws_files/NahuTestR11/NBA/CAS/InputFiles/CAS_CADPS2.annovar.avinput>

NOTICE: Running with system command <convert2annovar.pl  -includeinfo -allsample -withfreq -format vcf4 /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AAC/InputFiles/AAC_CADPS2.vcf.gz > /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AAC/InputFiles/AAC_CADPS2.annovar.avinput>

NOTICE: Running with system command <convert2annovar.pl  -includeinfo -allsample -withfreq -format vcf4 /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AFR/InputFiles/AFR_CADPS2.vcf.gz > /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AFR/InputFiles/AFR_CADPS2.annovar.avinput>

NOTICE: Running with system command <convert2annovar.pl  -includeinfo -allsample -withfreq -format vcf4 /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AJ/


SUCCESS: CAS, CADPS2

Extracting CADPS2 from EAS NBA Release 11 data.



NOTICE: Running with system command <convert2annovar.pl  -includeinfo -allsample -withfreq -format vcf4 /home/jupyter/workspace/ws_files/NahuTestR11/NBA/EAS/InputFiles/EAS_CADPS2.vcf.gz > /home/jupyter/workspace/ws_files/NahuTestR11/NBA/EAS/InputFiles/EAS_CADPS2.annovar.avinput>
NOTICE: Database index loaded. Total number of bins is 2858148 and the number of bins to be scanned is 561
NOTICE: Scanning filter database /home/jupyter/tools/annovar/humandb/hg38_gnomad41_genome.txt...


SUCCESS: AAC, CADPS2

Extracting CADPS2 from FIN NBA Release 11 data.


...


SUCCESS: CAH, CADPS2



NOTICE: Processing next batch with 2680 unique variants in 2680 input lines
NOTICE: Database index loaded. Total number of bins is 2858148 and the number of bins to be scanned is 555
DoneCE: Scanning filter database /home/jupyter/tools/annovar/humandb/hg38_gnomad41_genome.txt...
-----------------------------------------------------------------
NOTICE: Processing operation=f protocol=gnomad41_exome
NOTICE: Finished reading 18 column headers for '-dbtype gnomad41_exome'

NOTICE: Running system command <annotate_variation.pl -filter -dbtype gnomad41_exome -buildver hg38 -outfile /home/jupyter/workspace/ws_files/NahuTestR11/NBA/EAS/InputFiles/EAS_CADPS2.annovar /home/jupyter/workspace/ws_files/NahuTestR11/NBA/EAS/InputFiles/EAS_CADPS2.annovar.avinput /home/jupyter/tools/annovar/humandb/ -otherinfo>
NOTICE: Output file with variants matching filtering criteria is written to /home/jupyter/workspace/ws_files/NahuTestR11/NBA/EAS/InputFiles/EAS_CADPS2.annovar.hg38_gnomad41_exome_dropped, and ou


SUCCESS: EAS, CADPS2



NOTICE: Processing next batch with 6942 unique variants in 6942 input lines
NOTICE: Reading FASTA sequences from /home/jupyter/tools/annovar/humandb/hg38_refGeneMrna.fa ... Done with 17 sequences
-----------------------------------------------------------------
NOTICE: Processing operation=f protocol=clinvar_20250721
NOTICE: Finished reading 13 column headers for '-dbtype clinvar_20250721'

NOTICE: Running system command <annotate_variation.pl -filter -dbtype clinvar_20250721 -buildver hg38 -outfile /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AFR/InputFiles/AFR_CADPS2.annovar /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AFR/InputFiles/AFR_CADPS2.annovar.avinput /home/jupyter/tools/annovar/humandb/ -otherinfo>
NOTICE: the --dbtype clinvar_20250721 is assumed to be in generic ANNOVAR database format
NOTICE: Output file with variants matching filtering criteria is written to /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AFR/InputFiles/AFR_CADPS2.annovar.hg38_clinvar_20250721_d


SUCCESS: AFR, CADPS2



In [13]:
for ANCESTRY in ANCESTRIES:
    anno_dir = f"{ANA_DIR}/{ANCESTRY}/InputFiles"

    print(f"Working on {ANCESTRY}")
    
    df = pd.read_csv(f'{anno_dir}/{ANCESTRY}_CADPS2.annovar.hg38_multianno.txt', sep='\t')
    subset = df[["Chr", "Start", "End", "Ref", "Alt", "Func.refGene", "Gene.refGene", "ExonicFunc.refGene", "CLNSIG", "REVEL_score", "REVEL_rankscore", "CADD_raw", "CADD_raw_rankscore", "CADD_phred", "gnomad41_genome_fafmax_faf95_max"]]
    
    subset.to_csv(F'{anno_dir}/{ANCESTRY}_CADPS2.annovar.hg38_multianno_cropped.txt', sep='\t', index=False)
    
    print(f"{ANCESTRY} finished")

Working on FIN
FIN finished
Working on MDE
MDE finished
Working on SAS
SAS finished
Working on CAH
CAH finished


In [13]:
for GENE_NAME in GENE_DICT:
    for ANCESTRY in ANCESTRIES:
        print(f'WORKING ON: {ANCESTRY}')
        BUILD = "hg38"
        
        inputPrefix = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_{GENE_NAME}"
        # Read in ANNOVAR multianno file
        gene = pd.read_csv(f'{inputPrefix}.annovar.{BUILD}_multianno_cropped.txt', sep = '\t', header = 0)
            
        #Filter for the correct gene name (sometimes other genes are also included)
        gene['gnomad41_genome_fafmax_faf95_max'] = pd.to_numeric(gene['gnomad41_genome_fafmax_faf95_max'], errors='coerce')
        gene['CADD_phred'] = pd.to_numeric(gene['CADD_phred'], errors='coerce')
        gene['REVEL_score'] = pd.to_numeric(gene['REVEL_score'], errors='coerce')
        
            
        #Print number of variants in the different categories
        results = [] 
        
        intronic = gene[gene['Func.refGene']== 'intronic']
        upstream = gene[gene['Func.refGene']== 'upstream']
        downstream = gene[gene['Func.refGene']== 'downstream']
        utr5 = gene[gene['Func.refGene']== 'UTR5']
        utr3 = gene[gene['Func.refGene']== 'UTR3']
        splicing = gene[gene['Func.refGene']== 'splicing']
        exonic = gene[gene['Func.refGene']== 'exonic']
        stopgain = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'stopgain')]
        stoploss = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'stoploss')]
        startloss = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'startloss')]
        frameshift_deletion = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'frameshift deletion')]
        frameshift_insertion = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'frameshift insertion')]
        nonframeshift_deletion = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'nonframeshift deletion')]
        nonframeshift_insertion = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'nonframeshift insertion')]
        coding_nonsynonymous = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'nonsynonymous SNV')]
        coding_synonymous = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] == 'synonymous SNV')]
            
        #Predictors
        maf_01 = gene[gene['gnomad41_genome_fafmax_faf95_max']<0.01]
        maf_03 = gene[gene['gnomad41_genome_fafmax_faf95_max']<0.03]
        CADD_20 = gene[gene['CADD_phred']>20]
        REVEL_50 = gene[gene['REVEL_score']>0.5]
        lof = gene[(gene['Func.refGene'] == 'splicing') | (gene['ExonicFunc.refGene'] == 'stopgain') | (gene['ExonicFunc.refGene'] == 'stoploss') | (gene['ExonicFunc.refGene'] == 'startloss') | (gene['ExonicFunc.refGene'] == 'frameshift deletion') | (gene['ExonicFunc.refGene'] == 'frameshift insertion')]
        
        #CLNSIG annotation
        pathogenic =    gene[(gene['CLNSIG'] == 'Pathogenic')]
        likely_patho =  gene[(gene['CLNSIG'] == 'Likely_pathogenic')]
        var_unk_sig =   gene[(gene['CLNSIG'] == 'Uncertain_significance')]
        likely_benign = gene[(gene['CLNSIG'] == 'Likely_benign')]
        benign =        gene[(gene['CLNSIG'] == 'Benign')]
      
        variantsDict = {
                "Total variants": gene,
                "Intronic": intronic,
                "Upstream": upstream,
                "Downstream": downstream,
                "UTR3": utr3,
                "UTR5": utr5,
                "Splicing": splicing,
                "Exonic": exonic,
                "Stopgain": stopgain,
                "Stoploss": stoploss,
                "Frameshift deletion": frameshift_deletion,
                "Frameshift insertion": frameshift_insertion,
                "Non-frameshift insertion": nonframeshift_insertion,
                "Non-frameshift deletion": nonframeshift_deletion,
                "Synonymous": coding_synonymous,
                "Nonsynonymous": coding_nonsynonymous,
                "LOF": lof,
                "MAF01": maf_01,
                "MAF03": maf_03,
                "REVEL>0.5": REVEL_50,
                "CADD>20": CADD_20,
                "Pathogenic": pathogenic,
                "Likely Pathogenic": likely_patho,
                "Uncertain significance": var_unk_sig,
                "Likely Benign": likely_benign,
                "Benign": benign,
            }

                      
        variantSummary = pd.DataFrame({
                "variantType": list(variantsDict.keys()),
                "count": [len(variant) for variant in variantsDict.values()]
            })
        
        ## Save numbers of variants for future reference
        os.makedirs(f"{ANA_DIR}/{ANCESTRY}/VariantDescriptives", exist_ok = True)
        variantSummary.to_csv(f"{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}_variantCounts_refGene.csv", index=False)
       
        ## Print variants which are not "0"
        allvar = variantSummary
        allVariantSummary = variantSummary.to_string(index=False, header=False)
        filtered = variantSummary[variantSummary['count'] > 0]
        filteredVariantSummary = filtered.to_string(index=False, header=False)
        variantSummary_T = variantSummary.T.reset_index()
        variantSummary_T.columns = variantSummary_T.iloc[0]
        variantSummary_T = variantSummary_T[1:].reset_index(drop=True)
        descriptives = variantSummary_T.copy()
        descriptives.insert(0, 'Ancestry', ANCESTRY)
        descriptives.insert(1, 'Gene', GENE_NAME)
        descriptives.to_csv(f'{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}_variant_descriptives', sep="\t", index=False)
        filteredVariantSummary = filtered.to_string(index=False, header=False)                
               
        print(f"""
            =====================================
                   VARIANT SUMMARY FOR {ANCESTRY}
                   -----------------------
                   {variantSummary} 
            =====================================
                """)
            
        ## For rvtests
        ## Uncomment to_csv commands if you need to debug or full outputs
        all_variants = gene
        all_variants.to_csv(f"{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}_all_variants.csv", index=False)

        # Potential functional: These are variants annotated as frameshift, nonframeshift, startloss, stoploss, stopgain, splicing, missense, exonic, UTR5, UTR3, upstream (-100bp), downstream (+100bp), or ncRNA.
        potentially_functional = gene[gene['Func.refGene'] != 'intronic']
        potentially_functional.to_csv(f"{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}_variants_full_potentially_functional.csv", index=False)
                
        # Coding: These are variants annotated as frameshift, nonframeshift, startloss, stoploss, stopgain, splicing, or missense.
        coding_variants = gene[(gene['Func.refGene'] == 'exonic') & (gene['ExonicFunc.refGene'] != 'synonymous SNV')]
        coding_variants.to_csv(f"{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}_variants_full_coding.csv", index=False)

        potencially_pathogenic = gene[(gene['CLNSIG'] == 'Pathogenic') | (gene['CLNSIG'] == 'Likely_pathogenic') | (gene['CADD_phred']>20) | (gene['REVEL_score']>0.5)]
        potencially_pathogenic.to_csv(f"{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}_variants_potencially_pathogenic.csv", index=False)

        # Save in PLINK format
        variants_toKeep = potentially_functional[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
        variants_toKeep.to_csv(f'{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}.potentially_functional.variantstoKeep.txt', sep="\t", index=False, header=False)
            
        variants_toKeep2 = coding_variants[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
        variants_toKeep2.to_csv(f'{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}.coding_variants.variantstoKeep.txt', sep="\t", index=False, header=False)
            
        variants_toKeep4 = coding_nonsynonymous[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
        variants_toKeep4.to_csv(f'{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}.exonic.variantstoKeep.txt', sep="\t", index=False, header=False)
                
        variants_toKeep5 = all_variants[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
        variants_toKeep5.to_csv(f'{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}.all_variants.variantstoKeep.txt', sep="\t", index=False, header=False)

        variants_toKeep6 = maf_01[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
        variants_toKeep6.to_csv(f'{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}.maf01.variantstoKeep.txt', sep="\t", index=False, header=False)

        variants_toKeep7 = maf_03[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
        variants_toKeep7.to_csv(f'{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}.maf03.variantstoKeep.txt', sep="\t", index=False, header=False)

        variants_toKeep8 = lof[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
        variants_toKeep8.to_csv(f'{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}.lof.variantstoKeep.txt', sep="\t", index=False, header=False)

        variants_toKeep9 = potencially_pathogenic[['Chr', 'Start', 'End', 'Gene.refGene']].copy()
        variants_toKeep9.to_csv(f'{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}.potencially_pathogenic.variantstoKeep.txt', sep="\t", index=False, header=False)


WORKING ON: AAC

                   VARIANT SUMMARY FOR AAC
                   -----------------------
                                    variantType  count
0             Total variants   5309
1                   Intronic   5253
2                   Upstream      1
3                 Downstream      0
4                       UTR3     16
5                       UTR5      5
6                   Splicing      0
7                     Exonic     34
8                   Stopgain      1
9                   Stoploss      0
10       Frameshift deletion      1
11      Frameshift insertion      0
12  Non-frameshift insertion      0
13   Non-frameshift deletion      0
14                Synonymous     13
15             Nonsynonymous     19
16                       LOF      2
17                     MAF01   3578
18                     MAF03   4504
19                 REVEL>0.5      2
20                   CADD>20     16
21                Pathogenic      0
22         Likely Pathogenic      0
23    Uncertai

In [14]:
all_variant_stats = []
patho_variant_stats = []
for GENE_NAME in GENE_DICT:
        for ANCESTRY in ANCESTRIES:
            descriptives = pd.read_csv(f'{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}_variant_descriptives', sep="\t", header=0)
            all_variant_stats.append(descriptives)
all_variant_stats = pd.concat(all_variant_stats)
all_variant_stats.to_csv(f'{RESU_DIR}/{analysis}/CADPS2_{analysis}_AllVariantsStats.csv', index=False)

## Burden Analyses using RVTests

In [13]:
#Burden tests are performed in three groups: maf < 0.01, maf < 0.03 and potencially pathogenic.

burden_class = ['maf01', 'maf03', 'potencially_pathogenic']
for i in burden_class:  
    print(i)

maf01
maf03
potencially_pathogenic


In [25]:
## Extract variants of interest as vcf

def burdenPrep(GENE_NAME, ANCESTRY, burden_class):

    ## Input files
    inputRawPFILE = f"{IMPUTED_GENO_PATH}/{ANCESTRY}/chr7_{ANCESTRY}_release11_vwb"
    samplesToKeep = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}.samplestokeep"
    variantsToKeep = f"{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}.{burden_class}.variantstoKeep.txt"
    outputVCF = f"{ANA_DIR}/{ANCESTRY}/Burden/{ANCESTRY}_{GENE_NAME}.{burden_class}"
    
    ## Define plink2 command for extraction
    variantExtractor = [
        f"{TOOL_DIR}/plink2",
        "--pfile", inputRawPFILE,
        "--keep", samplesToKeep,
        "--extract", "range", variantsToKeep,
        "--recode", "vcf-iid",
        "--out", outputVCF
    ]

        ## Run plink2 command as bash subprocess
    subprocess.run(variantExtractor, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)
    vcfGzip = [
        "bgzip",
        "-f", f"{outputVCF}.vcf"
        ]

        ## Run plink2 command as bash subprocess
    subprocess.run(vcfGzip, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)
    vcfIndex = [
        "tabix",
        "-f", "-p", "vcf", f"{outputVCF}.vcf.gz"      
        ]    
    ## Run plink2 command as bash subprocess
    subprocess.run(vcfIndex, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)

In [27]:
for ancestry in ANCESTRIES:
    print(f'WORKING ON: {ancestry}\n')
    for burden in burden_class: 
        print(f'\tClass: {burden}')
        try:
            burdenPrep("CADPS2", ancestry, burden)
            print(f'\t{ancestry}, {burden} - DONE\n')
        except Exception as e:
            print(f'{ancestry}, {burden} - FAIL\n')

WORKING ON: AAC

	Class: maf01
	AAC, maf01 - DONE

	Class: maf03
	AAC, maf03 - DONE

	Class: potencially_pathogenic
	AAC, potencially_pathogenic - DONE

WORKING ON: AFR

	Class: maf01
	AFR, maf01 - DONE

	Class: maf03
	AFR, maf03 - DONE

	Class: potencially_pathogenic
	AFR, potencially_pathogenic - DONE

WORKING ON: AJ

	Class: maf01
	AJ, maf01 - DONE

	Class: maf03
	AJ, maf03 - DONE

	Class: potencially_pathogenic
	AJ, potencially_pathogenic - DONE

WORKING ON: AMR

	Class: maf01
	AMR, maf01 - DONE

	Class: maf03
	AMR, maf03 - DONE

	Class: potencially_pathogenic
	AMR, potencially_pathogenic - DONE

WORKING ON: CAS

	Class: maf01
	CAS, maf01 - DONE

	Class: maf03
	CAS, maf03 - DONE

	Class: potencially_pathogenic
	CAS, potencially_pathogenic - DONE

WORKING ON: EUR

	Class: maf01
	EUR, maf01 - DONE

	Class: maf03
	EUR, maf03 - DONE

	Class: potencially_pathogenic
	EUR, potencially_pathogenic - DONE

WORKING ON: EAS

	Class: maf01
	EAS, maf01 - DONE

	Class: maf03
	EAS, maf03 - DONE

	

In [14]:
## Run burden test with RVtests

def burdenRVTests(GENE_NAME, ANCESTRY, burden_class):
   
    ## Get ANCESTRY name
    inputVCF = f"{ANA_DIR}/{ANCESTRY}/Burden/{ANCESTRY}_{GENE_NAME}.{burden_class}.vcf.gz"
    outputBURDEN = f"{ANA_DIR}/{ANCESTRY}/Burden/{ANCESTRY}_{GENE_NAME}_{burden_class}.burden"
    covariate = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_covariate_file.txt"
    refFlat = f"{TOOL_DIR}/refFlat_HG38_all_chr.txt"
    
    burdenRVTests = [
        #RVtests with covariates 
        #Make sure the pheno and covariate file starts with the first 5 columsn: fid, iid, fatid, matid, sex
        #The pheno-name flag only works when the pheno/covar file is structured properly
        f"{TOOL_DIR}/rvtests/executable/rvtest", 
        "--noweb", 
        "--hide-covar",
        "--out", outputBURDEN,
        "--kernel", "skat,skato",
        "--inVcf", inputVCF,
        "--pheno", covariate,
        "--pheno-name", "PHENO",
        "--gene", GENE_NAME,
        "--geneFile", refFlat,
        "--covar", covariate,
        "--covar-name", "SEX,AGE,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10",
    ]
    
    ## Run plink2 command as bash subprocess
    subprocess.run(burdenRVTests, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)

In [ ]:
refFlat = f"{TOOL_DIR}/refFlat_HG38_all_chr.txt"
for ancestry in ANCESTRIES:
    print(f'WORKING ON: {ancestry}\n')
    for burden in burden_class: 
        print(f'\tClass: {burden}')
        try:
            burdenRVTests("CADPS2", ancestry, burden)
            print(f'\t{ancestry}, {burden} - DONE\n')
        except Exception as e:
            print(f'{ancestry}, {burden} - FAIL: {e}\n')

WORKING ON: EUR

	Class: maf01


In [14]:
skat_df = []
skatO_df = []

for GENE_NAME in GENE_DICT:
    for ANCESTRY in ANCESTRIES:
        for cls in burden_class:
        
            path_skat = Path(f"{ANA_DIR}/{ANCESTRY}/Burden/{ANCESTRY}_{GENE_NAME}_{cls}.burden.Skat.assoc")
            path_skatO = Path(f"{ANA_DIR}/{ANCESTRY}/Burden/{ANCESTRY}_{GENE_NAME}_{cls}.burden.SkatO.assoc")            
            
            try:
                if path_skat.is_file():
                    df = pd.read_csv(path_skat, sep="\t")
                    df.insert(0, "ANCESTRY", ANCESTRY)
                    df.insert(1, "GENE", GENE_NAME)
                    df.insert(2, "CLASS", cls)
                    skat_df.append(df)
            except:
                pass
            
            try:
                if path_skatO.is_file():
                    df = pd.read_csv(path_skatO, sep="\t")
                    df.insert(0, "ANCESTRY", ANCESTRY)
                    df.insert(1, "GENE", GENE_NAME)
                    df.insert(2, "CLASS", cls)
                    skatO_df.append(df)
            except:
                pass


In [15]:
skat_df = pd.concat(skat_df, ignore_index=True) if skat_df else pd.DataFrame()
skatO_df = pd.concat(skatO_df, ignore_index=True) if skatO_df else pd.DataFrame()

In [16]:
skat_df

,ANCESTRY,GENE,CLASS,Gene,RANGE,N_INFORMATIVE,NumVar,NumPolyVar,Q,Pvalue,NumPerm,ActualPerm,Stat,NumGreater,NumEqual,PermPvalue
0,AAC,CADPS2,maf01,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",1178,3408,3343,1.970880e+06,0.570715,10000,1598,1.970880e+06,1000,0,0.625782
1,AAC,CADPS2,maf03,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",1178,4298,4229,4.717270e+06,0.242942,10000,4052,4.717270e+06,1000,0,0.246792
2,AAC,CADPS2,potencially_pathogenic,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",1178,16,16,1.502670e+04,0.183620,10000,4740,1.502670e+04,1000,0,0.210970
3,AFR,CADPS2,maf01,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",6504,4950,4902,1.412050e+07,0.456168,10000,2118,1.412050e+07,1000,0,0.472144
4,AFR,CADPS2,maf03,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",6504,5732,5676,2.968870e+07,0.528860,10000,1751,2.968870e+07,1000,0,0.571102
5,AFR,CADPS2,potencially_pathogenic,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",6504,22,22,4.718570e+04,0.667842,10000,1537,4.718570e+04,1000,0,0.650618
6,AJ,CADPS2,maf01,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",1868,1484,1395,1.069670e+06,0.722846,10000,1312,1.069670e+06,1000,0,0.762195
7,AJ,CADPS2,maf03,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",1868,1822,1724,1.535850e+06,0.764162,10000,1269,1.535850e+06,1000,0,0.788022
8,AJ,CADPS2,potencially_pathogenic,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",1868,7,7,2.371640e+03,0.558081,10000,1770,2.371640e+03,1000,0,0.564972
9,AMR,CADPS2,maf01,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",3325,2429,2382,1.349010e+06,0.851333,10000,1133,1.349010e+06,1000,0,0.882613


In [17]:
skatO_df

,ANCESTRY,GENE,CLASS,Gene,RANGE,N_INFORMATIVE,NumVar,NumPolyVar,Q,rho,Pvalue
0,AAC,CADPS2,maf01,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",1178,3408,3343,5.876050e+07,1.0,0.156816
1,AAC,CADPS2,maf03,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",1178,4298,4229,2.038630e+07,0.1,0.125826
2,AAC,CADPS2,potencially_pathogenic,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",1178,16,16,2.348090e+04,1.0,0.069439
3,AFR,CADPS2,maf01,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",6504,4950,4902,7.060250e+06,0.0,0.673830
4,AFR,CADPS2,maf03,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",6504,5732,5676,3.970740e+08,1.0,0.516480
5,AFR,CADPS2,potencially_pathogenic,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",6504,22,22,3.589250e+04,1.0,0.497383
6,AJ,CADPS2,maf01,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",1868,1484,1395,1.403840e+06,1.0,0.519768
7,AJ,CADPS2,maf03,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",1868,1822,1724,7.434410e+06,1.0,0.159639
8,AJ,CADPS2,potencially_pathogenic,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",1868,7,7,1.303330e+03,1.0,0.537736
9,AMR,CADPS2,maf01,CADPS2,"7:122318423-122886460,7:122318423-122886460,7:...",3325,2429,2382,8.924940e+06,1.0,0.234360


In [18]:
skat_df.to_csv(f"{RESU_DIR}/{analysis}/CADPS2.{analysis}_BURDEN.SKAT.tsv", sep='\t', index=False)
skatO_df.to_csv(f"{RESU_DIR}/{analysis}/CADPS2.{analysis}_BURDEN.SKAT-O.tsv", sep='\t', index=False)

## Case/Control Analysis

### Glossary

- CHR Chromosome code
- SNP Variant identifier
- A1 Allele 1 (usually minor)
- A2 Allele 2 (usually major)
- MAF Allele 1 frequency in all subjects
- F_A/MAF_A Allele 1 frequency in cases
- F_U/MAF_U Allele 1 frequency in controls
- NCHROBS_A Number of case allele observations
- NCHROBS_U Number of control allele observations

## ALL VARIANTS


#### assoc

In [89]:
ANCESTRIES = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']

for anc in ANCESTRIES:
    print(f"Working on {anc}")
    
    inputpfile = f"{ANA_DIR}/{anc}/InputFiles/{anc}_CADPS2"
    outbfile = f"{ANA_DIR}/{anc}/InputFiles/{anc}_CADPS2"
    
    makebfiles = [
        f"{TOOL_DIR}/plink2",
          "--pfile", inputpfile,
          "--make-bed",
          "--out", outbfile
    ]
    
    subprocess.run(makebfiles, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)

    print(f"{anc} done, saved in {ANA_DIR}/{anc}/InputFiles/{anc}_CADPS2")

Working on AAC
AAC done, saved in /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AAC/InputFiles/AAC_CADPS2
Working on AFR
AFR done, saved in /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AFR/InputFiles/AFR_CADPS2
Working on AJ
AJ done, saved in /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AJ/InputFiles/AJ_CADPS2
Working on AMR
AMR done, saved in /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AMR/InputFiles/AMR_CADPS2
Working on CAS
CAS done, saved in /home/jupyter/workspace/ws_files/NahuTestR11/NBA/CAS/InputFiles/CAS_CADPS2
Working on EAS
EAS done, saved in /home/jupyter/workspace/ws_files/NahuTestR11/NBA/EAS/InputFiles/EAS_CADPS2
Working on EUR
EUR done, saved in /home/jupyter/workspace/ws_files/NahuTestR11/NBA/EUR/InputFiles/EUR_CADPS2
Working on FIN
FIN done, saved in /home/jupyter/workspace/ws_files/NahuTestR11/NBA/FIN/InputFiles/FIN_CADPS2
Working on MDE
MDE done, saved in /home/jupyter/workspace/ws_files/NahuTestR11/NBA/MDE/InputFiles/MDE_CADPS2
Working on SAS
SAS done

In [90]:
## extract region using plink

def variantAssoc(ANCESTRY):
    
    ## Analysis start time
    ts_start = datetime.now().strftime('Date: %Y-%m-%d   Time: %H-%M-%S')
    print(f"\n\n=============   ASSOCIATION START:   {ts_start}   =====================\n")
    
    ## Get ANCESTRY name
    print(f"Association test for {GENE_NAME} from {ANCESTRY} NBA Release 11 data.")

    samplesToKeep = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}.samplestokeep"
    inputbfile = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_{GENE_NAME}"
    outputAssoc = f"{ANA_DIR}/{ANCESTRY}/Association/{ANCESTRY}_{GENE_NAME}.assoc.allVariants"
    outputRaw = f"{ANA_DIR}/{ANCESTRY}/Association/{ANCESTRY}_{GENE_NAME}.raw.allVariants"
    covariate = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_covariate_file.txt"

    os.makedirs(f"{ANA_DIR}/{ANCESTRY}/Association", exist_ok = True)

    plinkAssoc = [
        f"{TOOL_DIR}/plink1.9",
        "--bfile", inputbfile,
        "--keep", samplesToKeep,
        "--assoc",
        "--maf", "0.01",
        "--mac", "2",
        "--hwe", "0.0001",
        "--adjust",
        "--ci", "0.95",
        "--out", outputAssoc
    ]

    subprocess.run(plinkAssoc, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)

    recodeA = [
            f"{TOOL_DIR}/plink1.9",
            "--bfile", inputbfile,
            "--keep", samplesToKeep,
            "--recode", "A",
            "--out", outputRaw
        ]

    subprocess.run(recodeA, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)
    
    ## Analysis end time
    ts_finish = datetime.now().strftime('Date: %Y-%m-%d   Time: %H-%M-%S')
    print(f"\n\n=============   ASSOCIATION END:   {ts_finish}   =====================\n")
    
    ## Return ANCESTRY, position and gene
    return f"Test performed on: {ANCESTRY} in {GENE_NAME}"

In [91]:
for ances in ANCESTRIES:
    variantAssoc(ances)



=============   ASSOCIATION START:   Date: 2026-02-25   Time: 19-32-44   =====================

Association test for CADPS2 from AAC NBA Release 11 data.


=============   ASSOCIATION END:   Date: 2026-02-25   Time: 19-32-46   =====================



=============   ASSOCIATION START:   Date: 2026-02-25   Time: 19-32-46   =====================

Association test for CADPS2 from AFR NBA Release 11 data.


=============   ASSOCIATION END:   Date: 2026-02-25   Time: 19-32-51   =====================



=============   ASSOCIATION START:   Date: 2026-02-25   Time: 19-32-51   =====================

Association test for CADPS2 from AJ NBA Release 11 data.


=============   ASSOCIATION END:   Date: 2026-02-25   Time: 19-32-54   =====================



=============   ASSOCIATION START:   Date: 2026-02-25   Time: 19-32-54   =====================

Association test for CADPS2 from AMR NBA Release 11 data.


=============   ASSOCIATION END:   Date: 2026-02-25   Time: 19-32-57   ================

In [ ]:
#Process results from plink assoc unadjusted analysis for CODING variants
#As there are very few or no significant variants with p-value < 0.05 - we will save results dataframe of all coding variants

for ANCESTRY in ANCESTRIES:
    ## Input files
    outputAssoc = f"{ANA_DIR}/{ANCESTRY}/Association/{ANCESTRY}_{GENE_NAME}.assoc.allVariants"
    outputRaw = f"{ANA_DIR}/{ANCESTRY}/Association/{ANCESTRY}_{GENE_NAME}.raw.allVariants"

    #Look at assoc results
    freq = pd.read_csv(f'{outputAssoc}.assoc', sep='\s+')
    
    #Filter for significant variants p < 0.05 - if any
    sig_all_nonadj = freq[freq['P']<0.05]
    
    print(f'There are {len(sig_all_nonadj)} variants with p-value < 0.05')

    #Read in plink recoded data (.raw file)
    recode = pd.read_csv(f'{outputRaw}.raw', sep='\s+')

    # Make a list from the column names
    column_names = recode.columns.tolist()

    # Drop the first 6 columns to keep the variants 
    variants = column_names[6:]

    print(f'Number of variants in {ANCESTRY} for {GENE_NAME}: {len(variants)}')

    # Pre-filter the dataset
    cases_data = recode[recode['PHENOTYPE'] == 2]
    controls_data = recode[recode['PHENOTYPE'] == 1]

    results = []

    # Pre-filter the dataset
    total_cases = cases_data.shape[0]
    total_controls = controls_data.shape[0]
    results = []

    for variant in variants:
        ## For PD cases
        hom_cases = (cases_data[variant] == 2).sum()
        het_cases = (cases_data[variant] == 1).sum()
        hom_ref_cases = (cases_data[variant] == 0).sum()  # Homozygous reference genotype
        missing_cases = total_cases - (hom_cases + het_cases + hom_ref_cases)  # Missing data count
        freq_cases = (2 * hom_cases + het_cases) / (2 * (total_cases - missing_cases))  # Adjust for missing data in denominator

        ## For controls
        hom_controls = (controls_data[variant] == 2).sum()
        het_controls = (controls_data[variant] == 1).sum()
        hom_ref_controls = (controls_data[variant] == 0).sum()  # Homozygous reference genotype
        missing_controls = total_controls - (hom_controls + het_controls + hom_ref_controls)  # Missing data count
        freq_controls = (2 * hom_controls + het_controls) / (2 * (total_controls - missing_controls))  # Adjust for missing data in denominator
    
        # Append results in dictionary format
        results.append({
            'Variant': variant,
            'Hom Cases': hom_cases,
            'Het Cases': het_cases,
            'Hom Ref Cases': hom_ref_cases,
            'Missing Cases': missing_cases,
            'Total Cases': total_cases,
            'Carrier Freq in Cases': freq_cases,
            'Hom Controls': hom_controls,
            'Het Controls': het_controls,
            'Hom Ref Controls': hom_ref_controls,
            'Missing Controls': missing_controls,
            'Total Controls': total_controls,
            'Carrier Freq in Controls': freq_controls
        })

    # Return
    df_results = pd.DataFrame(results)
    df_results['SNP'] = df_results['Variant'].apply(lambda x: x.rsplit('_', 1)[0])

    #Print dimensions of the df_results dataframe
    print(f'df_results shape: {df_results.shape}')
          
    #Merge with the assoc file
    sig_merge = freq[['SNP','A1','F_A','F_U','A2','L95','OR', 'SE', 'U95','P']]
    merged = pd.merge(df_results, sig_merge, on='SNP', how='right')
    
    #Print dimensions of the merged dataframe (just adding more columns)
    print(f'Merged dataframe shape: {merged.shape}\n\n') 
    
    ## Save to CSV
    merged.to_csv(f'{ANA_DIR}/{ANCESTRY}/{ANCESTRY}_{GENE_NAME}.assoc.allVariants.txt', sep = '\t', index=False)

There are 84 variants with p-value < 0.05
Number of variants in AAC for CADPS2: 5309
df_results shape: (5309, 14)
Merged dataframe shape: (1178, 23)


There are 21 variants with p-value < 0.05
Number of variants in AFR for CADPS2: 6942
df_results shape: (6942, 14)
Merged dataframe shape: (1059, 23)


There are 17 variants with p-value < 0.05
Number of variants in AJ for CADPS2: 2665
df_results shape: (2665, 14)
Merged dataframe shape: (510, 23)


There are 308 variants with p-value < 0.05
Number of variants in AMR for CADPS2: 4720
df_results shape: (4720, 14)
Merged dataframe shape: (423, 23)


There are 31 variants with p-value < 0.05
Number of variants in CAS for CADPS2: 2522
df_results shape: (2522, 14)
Merged dataframe shape: (398, 23)


There are 69 variants with p-value < 0.05
Number of variants in EAS for CADPS2: 2680
df_results shape: (2680, 14)
Merged dataframe shape: (365, 23)


There are 67 variants with p-value < 0.05


#### GLM

In [14]:
## extract region using plink

def variantGLM(ANCESTRY):
    
    ## Analysis start time
    ts_start = datetime.now().strftime('Date: %Y-%m-%d   Time: %H-%M-%S')
    print(f"\n\n=============   GLM START:   {ts_start}   =====================\n")
    
    ## Get ANCESTRY name
    print(f"GLM test for {GENE_NAME} from {ANCESTRY} NBA Release 11 data.")

    inputpfile = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_{GENE_NAME}"
    outputGLM = f"{ANA_DIR}/{ANCESTRY}/GLM/{ANCESTRY}_{GENE_NAME}.glm.allVariants"
    outputRaw = f"{ANA_DIR}/{ANCESTRY}/GLM/{ANCESTRY}_{GENE_NAME}.raw.allVariants"
    samplesToKeep = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}.samplestokeep"
    covariate = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_covariate_file.txt"
    # Print the command to be executed (for debugging purposes)
    print(f'Running Plink2 GLM for ANCESTRY: {ANCESTRY}')
    
    ## Define plink2 command for extraction
    plinkGLM = [
        f"{TOOL_DIR}/plink2",
        "--pfile", inputpfile,
        "--keep", samplesToKeep,
        "--glm",
        "--maf", "0.01",
        "--mac", "2",
        "--hwe", "0.0001", "keep-fewhet",
        "--adjust",
        "--allow-no-sex",
        "--ci", "0.95",
        "--covar", covariate,
        "--covar-variance-standardize",
        "--covar-name", "SEX,AGE,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10",
        "--out", outputGLM
    ]

    ## Run plink2 command as bash subprocess
    subprocess.run(plinkGLM, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)

    recodeA = [
            f"{TOOL_DIR}/plink2",
            "--pfile", inputpfile,
            "--keep", samplesToKeep,
            "--recode", "A",
            "--out", outputRaw
        ]

    subprocess.run(recodeA, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)
    
    ## Analysis end time
    ts_finish = datetime.now().strftime('Date: %Y-%m-%d   Time: %H-%M-%S')
    print(f"\n\n=============   GLM END:   {ts_finish}   =====================\n")
    
    ## Return ANCESTRY, position and gene
    return f"GLM test performed on: {ANCESTRY} in {GENE_NAME}"

In [15]:
for anc in ANCESTRIES:
    os.makedirs(f"{ANA_DIR}/{anc}/GLM", exist_ok = True)
    variantGLM(anc)



=============   GLM START:   Date: 2026-02-25   Time: 16-51-37   =====================

GLM test for CADPS2 from AAC NBA Release 11 data.
Running Plink2 GLM for ANCESTRY: AAC


=============   GLM END:   Date: 2026-02-25   Time: 16-51-40   =====================



=============   GLM START:   Date: 2026-02-25   Time: 16-51-40   =====================

GLM test for CADPS2 from AFR NBA Release 11 data.
Running Plink2 GLM for ANCESTRY: AFR


=============   GLM END:   Date: 2026-02-25   Time: 16-51-46   =====================



=============   GLM START:   Date: 2026-02-25   Time: 16-51-46   =====================

GLM test for CADPS2 from AJ NBA Release 11 data.
Running Plink2 GLM for ANCESTRY: AJ


=============   GLM END:   Date: 2026-02-25   Time: 16-51-48   =====================



=============   GLM START:   Date: 2026-02-25   Time: 16-51-48   =====================

GLM test for CADPS2 from AMR NBA Release 11 data.
Running Plink2 GLM for ANCESTRY: AMR


=============   GLM END:   D

In [22]:
for ANCESTRY in ANCESTRIES:
    try:
        glm_file = f"{ANA_DIR}/{ANCESTRY}/GLM/{ANCESTRY}_CADPS2.glm.allVariants.PHENO1.glm.logistic.hybrid"
        df = pd.read_csv(glm_file, sep='\t') 
        df = df.rename(columns={'LOG(OR)_SE': 'SE'})
        df.to_csv(glm_file, sep='\t', index=False)
    except:
        pass

In [12]:
#Process results from plink glm analysis for ALL variants
#As there are very few or no significant variants with p-value < 0.05 - we will save results dataframe of all coding variants

for ANCESTRY in ANCESTRIES:
    print(f'WORKING ON: {ANCESTRY}')    
    try:
        glm_file = f"{ANA_DIR}/{ANCESTRY}/GLM/{ANCESTRY}_{GENE_NAME}.glm.allVariants.PHENO1.glm.logistic.hybrid"
        raw_file = f"{ANA_DIR}/{ANCESTRY}/GLM/{ANCESTRY}_{GENE_NAME}.raw.allVariants.raw"

        glm = pd.read_csv(glm_file, sep='\s+')
        glm_add = glm[glm['TEST']=="ADD"]
        
        #Filter for significant variants p < 0.05 - if any
        significant = glm_add[glm_add['P']<0.05]
        print(f'There are {len(significant)} variants with p-value < 0.05 in glm')
        
        #Read in plink recoded data (.raw file)
        recode = pd.read_csv(raw_file, sep='\s+')
    
        # Make a list from the column names
        column_names = recode.columns.tolist()
    
        # Drop the first 6 columns to keep the variants 
        variants = column_names[6:]
    
        print(f'Number of variants in {ANCESTRY} for {GENE_NAME}: {len(variants)}')
    
        # Pre-filter the dataset
        cases_data = recode[recode['PHENOTYPE'] == 2]
        controls_data = recode[recode['PHENOTYPE'] == 1]
    
        results = []
    
        # Pre-filter the dataset
        total_cases = cases_data.shape[0]
        total_controls = controls_data.shape[0]
        results = []
    
        for variant in variants:
            ## For PD cases
            hom_cases = (cases_data[variant] == 2).sum()
            het_cases = (cases_data[variant] == 1).sum()
            hom_ref_cases = (cases_data[variant] == 0).sum()  # Homozygous reference genotype
            missing_cases = total_cases - (hom_cases + het_cases + hom_ref_cases)  # Missing data count
            freq_cases = (2 * hom_cases + het_cases) / (2 * (total_cases - missing_cases))  # Adjust for missing data in denominator
    
            ## For controls
            hom_controls = (controls_data[variant] == 2).sum()
            het_controls = (controls_data[variant] == 1).sum()
            hom_ref_controls = (controls_data[variant] == 0).sum()  # Homozygous reference genotype
            missing_controls = total_controls - (hom_controls + het_controls + hom_ref_controls)  # Missing data count
            freq_controls = (2 * hom_controls + het_controls) / (2 * (total_controls - missing_controls))  # Adjust for missing data in denominator
        
            # Append results in dictionary format
            results.append({
                'Variant': variant,
                'Hom Cases': hom_cases,
                'Het Cases': het_cases,
                'Hom Ref Cases': hom_ref_cases,
                'Missing Cases': missing_cases,
                'Total Cases': total_cases,
                'Carrier Freq in Cases': freq_cases,
                'Hom Controls': hom_controls,
                'Het Controls': het_controls,
                'Hom Ref Controls': hom_ref_controls,
                'Missing Controls': missing_controls,
                'Total Controls': total_controls,
                'Carrier Freq in Controls': freq_controls
            })
    
        # Return
        df_results = pd.DataFrame(results)
        df_results['ID'] = df_results['Variant'].apply(lambda x: x.rsplit('_', 1)[0])
    
        #Print dimensions of the df_results dataframe
        print(f'df_results shape: {df_results.shape}')
        
        #Merge with the glm file
        sig_merge = glm_add[['ID','A1','A1_FREQ','OBS_CT','OR','SE','Z_STAT','P']]
        merged = pd.merge(df_results, sig_merge, on='ID', how='right')
        
        #Print dimensions of the merged dataframe (just adding more columns)
        print(f'Merged dataframe shape: {merged.shape}\n\n')

        ## Save to CSV
        merged.to_csv(f'{ANA_DIR}/{ANCESTRY}/{ANCESTRY}_{GENE_NAME}.glm.allVariants.txt', sep = '\t', index=False)
    except Exception as error:
        print(error)
        pass

WORKING ON: AAC
There are 53 variants with p-value < 0.05 in glm
Number of variants in AAC for CADPS2: 5309
df_results shape: (5309, 14)
Merged dataframe shape: (1203, 21)


WORKING ON: AFR
There are 38 variants with p-value < 0.05 in glm
Number of variants in AFR for CADPS2: 6942
df_results shape: (6942, 14)
Merged dataframe shape: (1120, 21)


WORKING ON: AJ
There are 6 variants with p-value < 0.05 in glm
Number of variants in AJ for CADPS2: 2665
df_results shape: (2665, 14)
Merged dataframe shape: (519, 21)


WORKING ON: AMR
There are 36 variants with p-value < 0.05 in glm
Number of variants in AMR for CADPS2: 4720
df_results shape: (4720, 14)
Merged dataframe shape: (439, 21)


WORKING ON: CAS
There are 12 variants with p-value < 0.05 in glm
Number of variants in CAS for CADPS2: 2522
df_results shape: (2522, 14)
Merged dataframe shape: (440, 21)


WORKING ON: EAS
There are 8 variants with p-value < 0.05 in glm
Number of variants in EAS for CADPS2: 2680
df_results shape: (2680, 14)


## EXONIC VARIANTS

#### assoc

In [13]:
## extract region using plink

def exonicAssoc(ANCESTRY):
    
    ## Analysis start time
    ts_start = datetime.now().strftime('Date: %Y-%m-%d   Time: %H-%M-%S')
    print(f"\n\n=============   ASSOC START:   {ts_start}   =====================\n")

    ## Get ANCESTRY name
    print(f"Association test for {GENE_NAME} from {ANCESTRY} {analysis} Release 11 data.")
    
    ## Input files
    inputBfile = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_{GENE_NAME}"
    samplesToKeep = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}.samplestokeep"
    covariate = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_covariate_file.txt"
    variantsToKeep = f'{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}.exonic.variantstoKeep.txt'
    outputAssocExonic = f"{ANA_DIR}/{ANCESTRY}/Association/{ANCESTRY}_{GENE_NAME}.assoc.allExonic"
    outputRawExonic = f"{ANA_DIR}/{ANCESTRY}/Association/{ANCESTRY}_{GENE_NAME}.raw.allExonic"
    
    # Print the command to be executed (for debugging purposes)
    print(f'Running Plink1.9 Association for ANCESTRY: {ANCESTRY}')
        
    ## Define plink2 command for extraction
    plinkAssocExonic = [
        f"{TOOL_DIR}/plink1.9",
        "--bfile", inputBfile,
        "--keep", samplesToKeep,
        "--extract", "range", variantsToKeep,
        "--assoc",
        "--maf", "0.01",
        "--mac", "2",
        "--hwe", "0.0001",
        "--adjust",
        "--allow-no-sex",
        "--ci", "0.95",
        "--out", outputAssocExonic
    ]

    ## Run plink2 command as bash subprocess
    subprocess.run(plinkAssocExonic, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)

    recodeA = [
        f"{TOOL_DIR}/plink1.9",
        "--bfile", inputBfile,
        "--keep", samplesToKeep,
        "--extract", "range", variantsToKeep,
        "--recode", "A",
        "--out", outputRawExonic
        ]

    subprocess.run(recodeA, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)
    
    ## Analysis end time
    ts_finish = datetime.now().strftime('Date: %Y-%m-%d   Time: %H-%M-%S')
    print(f"\n\n=============   ASSOC END:   {ts_finish}   =====================\n")
    
    ## Return ANCESTRY, position and gene
    return f"Association test performed on exonic variants: {ANCESTRY} in {GENE_NAME}"

In [14]:
for ances in ANCESTRIES:
    exonicAssoc(ances)



=============   ASSOC START:   Date: 2026-02-25   Time: 17-55-14   =====================

Association test for CADPS2 from AAC NBA Release 11 data.
Running Plink1.9 Association for ANCESTRY: AAC


=============   ASSOC END:   Date: 2026-02-25   Time: 17-55-16   =====================



=============   ASSOC START:   Date: 2026-02-25   Time: 17-55-16   =====================

Association test for CADPS2 from AFR NBA Release 11 data.
Running Plink1.9 Association for ANCESTRY: AFR


=============   ASSOC END:   Date: 2026-02-25   Time: 17-55-18   =====================



=============   ASSOC START:   Date: 2026-02-25   Time: 17-55-18   =====================

Association test for CADPS2 from AJ NBA Release 11 data.
Running Plink1.9 Association for ANCESTRY: AJ


=============   ASSOC END:   Date: 2026-02-25   Time: 17-55-19   =====================



=============   ASSOC START:   Date: 2026-02-25   Time: 17-55-19   =====================

Association test for CADPS2 from AMR NBA Release 

In [15]:
#Process results from plink assoc unadjusted analysis for CODING variants
#As there are very few or no significant variants with p-value < 0.05 - we will save results dataframe of all coding variants
ANCESTRIES = ['AFR', 'AAC', 'AJ', 'AMR', 'CAS', 'EUR', 'FIN', 'EAS', 'MDE', 'SAS', 'CAH']

for ANCESTRY in ANCESTRIES:
    try:
        ## Input files
        AssocExonic = f"{ANA_DIR}/{ANCESTRY}/Association/{ANCESTRY}_{GENE_NAME}.assoc.allExonic.assoc"
        RawExonic = f"{ANA_DIR}/{ANCESTRY}/Association/{ANCESTRY}_{GENE_NAME}.raw.allExonic.raw"
        
        #Look at assoc results
        freq = pd.read_csv(AssocExonic, sep='\s+')
        
        #Filter for significant variants p < 0.05 - if any
        sig_all_nonadj = freq[freq['P']<0.05]
        
        print(f'There are {len(sig_all_nonadj)} variants with p-value < 0.05')
    
        #Read in plink recoded data (.raw file)
        recode = pd.read_csv(RawExonic, sep='\s+')
    
        # Make a list from the column names
        column_names = recode.columns.tolist()
    
        # Drop the first 6 columns to keep the variants 
        variants = column_names[6:]
    
        print(f'Number of variants in {ANCESTRY} for CADPS2: {len(variants)}')
    
        # Pre-filter the dataset
        cases_data = recode[recode['PHENOTYPE'] == 2]
        controls_data = recode[recode['PHENOTYPE'] == 1]
    
        results = []
    
        # Pre-filter the dataset
        total_cases = cases_data.shape[0]
        total_controls = controls_data.shape[0]
        results = []
    
        for variant in variants:
            ## For PD cases
            hom_cases = (cases_data[variant] == 2).sum()
            het_cases = (cases_data[variant] == 1).sum()
            hom_ref_cases = (cases_data[variant] == 0).sum()  # Homozygous reference genotype
            missing_cases = total_cases - (hom_cases + het_cases + hom_ref_cases)  # Missing data count
            freq_cases = (2 * hom_cases + het_cases) / (2 * (total_cases - missing_cases))  # Adjust for missing data in denominator
    
            ## For controls
            hom_controls = (controls_data[variant] == 2).sum()
            het_controls = (controls_data[variant] == 1).sum()
            hom_ref_controls = (controls_data[variant] == 0).sum()  # Homozygous reference genotype
            missing_controls = total_controls - (hom_controls + het_controls + hom_ref_controls)  # Missing data count
            freq_controls = (2 * hom_controls + het_controls) / (2 * (total_controls - missing_controls))  # Adjust for missing data in denominator
        
            # Append results in dictionary format
            results.append({
                'Variant': variant,
                'Hom Cases': hom_cases,
                'Het Cases': het_cases,
                'Hom Ref Cases': hom_ref_cases,
                'Missing Cases': missing_cases,
                'Total Cases': total_cases,
                'Carrier Freq in Cases': freq_cases,
                'Hom Controls': hom_controls,
                'Het Controls': het_controls,
                'Hom Ref Controls': hom_ref_controls,
                'Missing Controls': missing_controls,
                'Total Controls': total_controls,
                'Carrier Freq in Controls': freq_controls
            })
    
        # Return
        df_results = pd.DataFrame(results)
        df_results['SNP'] = df_results['Variant'].apply(lambda x: x.rsplit('_', 1)[0])
    
        #Print dimensions of the df_results dataframe
        print(f'df_results shape: {df_results.shape}')
              
        #Merge with the assoc file
        sig_merge = freq[['SNP','A1','F_A','F_U','A2','L95','OR','U95','P']]
        merged = pd.merge(df_results, sig_merge, on='SNP', how='right')
        
        #Print dimensions of the merged dataframe (just adding more columns)
        print(f'Merged dataframe shape: {merged.shape}\n\n') 
        
        ## Save to CSV
        merged.to_csv(f'{ANA_DIR}/{ANCESTRY}/{ANCESTRY}_{GENE_NAME}.assoc.allExonic.txt', sep = '\t', index=False)    
    except: pass

There are 0 variants with p-value < 0.05
Number of variants in AFR for CADPS2: 25
df_results shape: (25, 14)
Merged dataframe shape: (2, 22)


There are 0 variants with p-value < 0.05
Number of variants in AAC for CADPS2: 19
df_results shape: (19, 14)
Merged dataframe shape: (2, 22)


There are 0 variants with p-value < 0.05
Number of variants in AJ for CADPS2: 10
df_results shape: (10, 14)
Merged dataframe shape: (1, 22)


There are 2 variants with p-value < 0.05
Number of variants in AMR for CADPS2: 22
df_results shape: (22, 14)
Merged dataframe shape: (2, 22)


There are 0 variants with p-value < 0.05
Number of variants in CAS for CADPS2: 13
df_results shape: (13, 14)
Merged dataframe shape: (2, 22)


There are 0 variants with p-value < 0.05
Number of variants in EUR for CADPS2: 66
df_results shape: (66, 14)
Merged dataframe shape: (1, 22)


There are 0 variants with p-value < 0.05
Number of variants in MDE for CADPS2: 11
df_results shape: (11, 14)
Merged dataframe shape: (1, 22)




#### GLM

In [16]:
## extract region using plink

def exonicGLM(ANCESTRY):
    
    ## Analysis start time
    ts_start = datetime.now().strftime('Date: %Y-%m-%d   Time: %H-%M-%S')
    print(f"\n\n=============   GLM START:   {ts_start}   =====================\n")

    ## Get ANCESTRY name
    print(f"GLM test for {GENE_NAME} from {ANCESTRY} NBA Release 11 data.")


    inputpfile = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_{GENE_NAME}"
    outputGLMExonic = f"{ANA_DIR}/{ANCESTRY}/GLM/{ANCESTRY}_{GENE_NAME}.glm.allExonic"
    outputRawExonic = f"{ANA_DIR}/{ANCESTRY}/GLM/{ANCESTRY}_{GENE_NAME}.raw.allExonic"
    variantsToKeep = f'{ANA_DIR}/{ANCESTRY}/VariantDescriptives/{ANCESTRY}_{GENE_NAME}.exonic.variantstoKeep.txt'
    samplesToKeep = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}.samplestokeep"
    covariate = f"{ANA_DIR}/{ANCESTRY}/InputFiles/{ANCESTRY}_covariate_file.txt"
    
    # Print the command to be executed (for debugging purposes)
    print(f'Running Plink2 GLM for ANCESTRY: {ANCESTRY}')
    
    
    ## Define plink2 command for extraction
    plinkGLMExonic = [
        f"{TOOL_DIR}/plink2",
        "--pfile", inputpfile,
        "--keep", samplesToKeep,
        "--extract", "range", variantsToKeep,
        "--glm",
        "--maf", "0.01",
        "--mac", "2",
        "--hwe", "0.0001", "keep-fewhet",
        "--adjust",
        "--allow-no-sex",
        "--ci", "0.95",
        "--covar", covariate,
        "--covar-variance-standardize",
        "--covar-name", "SEX,AGE,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10",
        "--out", outputGLMExonic
    ]

    ## Run plink2 command as bash subprocess
    subprocess.run(plinkGLMExonic, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)

    recodeA = [
        f"{TOOL_DIR}/plink2",
        "--pfile", inputpfile,
        "--keep", samplesToKeep,
        "--extract", "range", variantsToKeep,
        "--recode", "A",
        "--out", outputRawExonic
        ]

    subprocess.run(recodeA, stderr=subprocess.PIPE, stdout=subprocess.PIPE, check=False)
    
    ## Analysis end time
    ts_finish = datetime.now().strftime('Date: %Y-%m-%d   Time: %H-%M-%S')
    print(f"\n\n=============   GLM END:   {ts_finish}   =====================\n")
    
    ## Return ANCESTRY, position and gene
    return f"GLM test performed on exonic variants: {ANCESTRY} in {GENE_NAME}"

In [17]:
for ances in ANCESTRIES:
    exonicGLM(ances)



=============   GLM START:   Date: 2026-02-25   Time: 17-55-39   =====================

GLM test for CADPS2 from AFR NBA Release 11 data.
Running Plink2 GLM for ANCESTRY: AFR


=============   GLM END:   Date: 2026-02-25   Time: 17-55-41   =====================



=============   GLM START:   Date: 2026-02-25   Time: 17-55-41   =====================

GLM test for CADPS2 from AAC NBA Release 11 data.
Running Plink2 GLM for ANCESTRY: AAC


=============   GLM END:   Date: 2026-02-25   Time: 17-55-42   =====================



=============   GLM START:   Date: 2026-02-25   Time: 17-55-42   =====================

GLM test for CADPS2 from AJ NBA Release 11 data.
Running Plink2 GLM for ANCESTRY: AJ


=============   GLM END:   Date: 2026-02-25   Time: 17-55-44   =====================



=============   GLM START:   Date: 2026-02-25   Time: 17-55-44   =====================

GLM test for CADPS2 from AMR NBA Release 11 data.
Running Plink2 GLM for ANCESTRY: AMR


=============   GLM END:   D

In [20]:
for ANCESTRY in ANCESTRIES:
    try:
        glm_file = f"{ANA_DIR}/{ANCESTRY}/GLM/{ANCESTRY}_CADPS2.glm.allExonic.PHENO1.glm.logistic.hybrid"
        df = pd.read_csv(glm_file, sep='\t') 
        df = df.rename(columns={'LOG(OR)_SE': 'SE'})
        df.to_csv(glm_file, sep='\t', index=False)
    except:
        pass

In [47]:
#Process results from plink glm analysis for EXONIC variants
#As there are very few or no significant variants with p-value < 0.05 - we will save results dataframe of all coding variants

ANCESTRIES = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']

for ANCESTRY in ANCESTRIES:
    try:
        print(f'WORKING ON: {ANCESTRY}')

        glm_file = f"{ANA_DIR}/{ANCESTRY}/GLM/{ANCESTRY}_{GENE_NAME}.glm.allExonic.PHENO1.glm.logistic.hybrid"
        raw_file = f"{ANA_DIR}/{ANCESTRY}/GLM/{ANCESTRY}_{GENE_NAME}.raw.allExonic.raw"

        
        #Read in glm results
        glm = pd.read_csv(glm_file, sep='\s+')
        glm_add = glm[glm['TEST']=="ADD"]
        
        #Filter for significant variants p < 0.05 - if any
        significant = glm_add[glm_add['P']<0.05]
        print(f'There are {len(significant)} variants with p-value < 0.05 in glm')
        
        #Read in plink recoded data (.raw file)
        recode = pd.read_csv(raw_file, sep='\s+')
    
        # Make a list from the column names
        column_names = recode.columns.tolist()
    
        # Drop the first 6 columns to keep the variants 
        variants = column_names[6:]
    
        print(f'Number of variants in {ANCESTRY} for CADPS2: {len(variants)}')
    
        # Pre-filter the dataset
        cases_data = recode[recode['PHENOTYPE'] == 2]
        controls_data = recode[recode['PHENOTYPE'] == 1]
    
        results = []
    
        # Pre-filter the dataset
        total_cases = cases_data.shape[0]
        total_controls = controls_data.shape[0]
        results = []
    
        for variant in variants:
            ## For PD cases
            hom_cases = (cases_data[variant] == 2).sum()
            het_cases = (cases_data[variant] == 1).sum()
            hom_ref_cases = (cases_data[variant] == 0).sum()  # Homozygous reference genotype
            missing_cases = total_cases - (hom_cases + het_cases + hom_ref_cases)  # Missing data count
            freq_cases = (2 * hom_cases + het_cases) / (2 * (total_cases - missing_cases))  # Adjust for missing data in denominator
    
            ## For controls
            hom_controls = (controls_data[variant] == 2).sum()
            het_controls = (controls_data[variant] == 1).sum()
            hom_ref_controls = (controls_data[variant] == 0).sum()  # Homozygous reference genotype
            missing_controls = total_controls - (hom_controls + het_controls + hom_ref_controls)  # Missing data count
            freq_controls = (2 * hom_controls + het_controls) / (2 * (total_controls - missing_controls))  # Adjust for missing data in denominator
        
            # Append results in dictionary format
            results.append({
                'Variant': variant,
                'Hom Cases': hom_cases,
                'Het Cases': het_cases,
                'Hom Ref Cases': hom_ref_cases,
                'Missing Cases': missing_cases,
                'Total Cases': total_cases,
                'Carrier Freq in Cases': freq_cases,
                'Hom Controls': hom_controls,
                'Het Controls': het_controls,
                'Hom Ref Controls': hom_ref_controls,
                'Missing Controls': missing_controls,
                'Total Controls': total_controls,
                'Carrier Freq in Controls': freq_controls
            })
    
        # Return
        df_results = pd.DataFrame(results)
        df_results['ID'] = df_results['Variant'].apply(lambda x: x.rsplit('_', 1)[0])
    
        #Print dimensions of the df_results dataframe
        print(f'df_results shape: {df_results.shape}')
        
        #Merge with the glm file
        sig_merge = glm_add[['ID','A1','A1_FREQ','OBS_CT','OR','SE','Z_STAT','P']]
        merged = pd.merge(df_results, sig_merge, on='ID', how='right')
        
        #Print dimensions of the merged dataframe (just adding more columns)
        print(f'Merged dataframe shape: {merged.shape}')
        
        ## Save to CSV
        merged.to_csv(f'{ANA_DIR}/{ANCESTRY}/{ANCESTRY}_{GENE_NAME}.glm.allExonic.txt', sep = '\t', index=False)   
    except Exception as e:
        print(e)
        pass

WORKING ON: AAC
There are 0 variants with p-value < 0.05 in glm
Number of variants in AAC for CADPS2: 19
df_results shape: (19, 14)
Merged dataframe shape: (2, 21)
WORKING ON: AFR
There are 0 variants with p-value < 0.05 in glm
Number of variants in AFR for CADPS2: 25
df_results shape: (25, 14)
Merged dataframe shape: (2, 21)
WORKING ON: AJ
There are 0 variants with p-value < 0.05 in glm
Number of variants in AJ for CADPS2: 10
df_results shape: (10, 14)
Merged dataframe shape: (1, 21)
WORKING ON: AMR
There are 0 variants with p-value < 0.05 in glm
Number of variants in AMR for CADPS2: 22
df_results shape: (22, 14)
Merged dataframe shape: (2, 21)
WORKING ON: CAS
There are 0 variants with p-value < 0.05 in glm
Number of variants in CAS for CADPS2: 13
df_results shape: (13, 14)
Merged dataframe shape: (2, 21)
WORKING ON: EAS
[Errno 2] No such file or directory: '/home/jupyter/workspace/ws_files/NahuTestR11/NBA/EAS/GLM/EAS_CADPS2.glm.allExonic.PHENO1.glm.logistic.hybrid'
WORKING ON: EUR
Th

## Summary

In [48]:
import shutil
import glob

classes1 = ('allVariants','allExonic')
ANCESTRIES = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']

for i in ANCESTRIES:
    current_dir = os.path.join(ANA_DIR, str(i))
    summary_dir = os.path.join(current_dir, "Summary")
    if not os.path.exists(summary_dir):
        os.makedirs(summary_dir)

    pattern = os.path.join(current_dir, f"{i}_*")
    for file_path in glob.glob(pattern):
        if os.path.isfile(file_path):
            file_name = os.path.basename(file_path)
            destination = os.path.join(summary_dir, file_name)
            shutil.move(file_path, destination)

In [49]:
for GENE_NAME in GENE_DICT:
    for ANCESTRY in ANCESTRIES:
        for class1 in classes1:
            try:
                print(f"Working on {ANCESTRY}, class: {class1}\n")
                ## Input files
                summary_dir = f"{ANA_DIR}/{ANCESTRY}/Summary"
                base_path = f'{summary_dir}/{ANCESTRY}_{GENE_NAME}'
                glm = f"{base_path}.glm.{class1}"
                assoc = f"{base_path}.assoc.{class1}"
                
                glm_df = pd.read_csv(f"{glm}.txt", sep='\t', header=0)
                glm_adj_df = pd.read_csv(f'{ANA_DIR}/{ANCESTRY}/GLM/{ANCESTRY}_CADPS2.glm.{class1}.PHENO1.glm.logistic.hybrid.adjusted', sep='\t', header=0)            
                glm_merge = pd.merge(glm_df, glm_adj_df[['ID', 'BONF']], how='inner', on='ID')
                
                assoc_df = pd.read_csv(f"{assoc}.txt", sep='\t', header=0)   
                assoc_adj_df = pd.read_csv(f'{ANA_DIR}/{ANCESTRY}/Association/{ANCESTRY}_CADPS2.assoc.{class1}.assoc.adjusted', sep='\s+', header=0)           
                assoc_merge = pd.merge(assoc_df, assoc_adj_df[['SNP', 'BONF']], how='inner', on='SNP')
                assoc_merge['ID'] = assoc_merge['SNP']
                assoc_merge['A1_FREQ'] = assoc_merge['F_A']

                os.makedirs(summary_dir, exist_ok = True)

                glm_out = f"{summary_dir}/{ANCESTRY}_{GENE_NAME}_{class1}.glm"
                assoc_out = f"{summary_dir}/{ANCESTRY}_{GENE_NAME}_{class1}.assoc"
                glm_merge.to_csv(glm_out, index=False)
                assoc_merge.to_csv(assoc_out, index=False)
                
            except Exception as e:
                print(e)
            pass

Working on AAC, class: allVariants

Working on AAC, class: allExonic

Working on AFR, class: allVariants

Working on AFR, class: allExonic

Working on AJ, class: allVariants

Working on AJ, class: allExonic

Working on AMR, class: allVariants

Working on AMR, class: allExonic

Working on CAS, class: allVariants

Working on CAS, class: allExonic

Working on EAS, class: allVariants

Working on EAS, class: allExonic

[Errno 2] No such file or directory: '/home/jupyter/workspace/ws_files/NahuTestR11/NBA/EAS/Summary//EAS_CADPS2.glm.allExonic.txt'
Working on EUR, class: allVariants

Working on EUR, class: allExonic

Working on FIN, class: allVariants

Working on FIN, class: allExonic

[Errno 2] No such file or directory: '/home/jupyter/workspace/ws_files/NahuTestR11/NBA/FIN/Summary//FIN_CADPS2.glm.allExonic.txt'
Working on MDE, class: allVariants

Working on MDE, class: allExonic

Working on SAS, class: allVariants

Working on SAS, class: allExonic

Working on CAH, class: allVariants

Workin

In [69]:
ANCESTRIES = ['AAC', 'AFR', 'AJ', 'AMR', 'CAS', 'EAS', 'EUR', 'FIN', 'MDE', 'SAS', 'CAH']

for ANCESTRY in  ANCESTRIES:
    for classes in classes1:
        try:
            summary_dir = f"{ANA_DIR}/{ANCESTRY}/Summary"
            base = f"{summary_dir}/{ANCESTRY}_CADPS2_{classes}"
            df_glm = pd.read_csv(f"{base}.glm", sep=',', header=0)
            df_assoc = pd.read_csv(f"{base}.assoc", sep=',', header=0) 

            df_glm.to_csv(f"{summary_dir}/{ANCESTRY}_CADPS2_{classes}_processed_glm.csv", index=False)
            df_assoc.to_csv(f"{summary_dir}/{ANCESTRY}_CADPS2_{classes}_processed_assoc.csv", index=False)              
        except:
            pass

summary_stats = []
analysis1 = {'glm', 'assoc'}
for ANCESTRY in ANCESTRIES:
    for classes in classes1:
        for a in analysis1:
            try:
                summary_dir = f"{ANA_DIR}/{ANCESTRY}/Summary"
                base = f"{summary_dir}/{ANCESTRY}_CADPS2_{classes}_processed_{a}.csv"
                print(f"Read {base}")
                df = pd.read_csv(base, header=0)
                df.insert(0, "Gene", GENE_NAME)
                df.insert(1, "Ancestry", ANCESTRY)
                df.insert(2, "Analysis", a)
                df.insert(3, "Variant type", classes)
                summary_stats.append(df)
            except Exception as e:
                print(e)
                pass
                    
summary_stats = pd.concat(summary_stats, ignore_index=True) if summary_stats else pd.DataFrame()
summary_stats

Read /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AAC/Summary/AAC_CADPS2_allVariants_processed_assoc.csv
Read /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AAC/Summary/AAC_CADPS2_allVariants_processed_glm.csv
Read /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AAC/Summary/AAC_CADPS2_allExonic_processed_assoc.csv
Read /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AAC/Summary/AAC_CADPS2_allExonic_processed_glm.csv
Read /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AFR/Summary/AFR_CADPS2_allVariants_processed_assoc.csv
Read /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AFR/Summary/AFR_CADPS2_allVariants_processed_glm.csv
Read /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AFR/Summary/AFR_CADPS2_allExonic_processed_assoc.csv
Read /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AFR/Summary/AFR_CADPS2_allExonic_processed_glm.csv
Read /home/jupyter/workspace/ws_files/NahuTestR11/NBA/AJ/Summary/AJ_CADPS2_allVariants_processed_assoc.csv
Read /home/jupyter/workspace/ws_files

,Gene,Ancestry,Analysis,Variant type,Variant,Hom Cases,Het Cases,Hom Ref Cases,Missing Cases,Total Cases,...,L95,OR,SE,U95,P,BONF,ID,A1_FREQ,OBS_CT,Z_STAT
0,CADPS2,AAC,assoc,allVariants,chr7:122318683:A:C_C,3,30,445,0,478,...,0.6128,0.9273,0.211300,1.403,0.720800,1.0000,chr7:122318683:A:C,0.037660,NaN,NaN
1,CADPS2,AAC,assoc,allVariants,chr7:122319215:G:T_T,0,13,462,3,478,...,0.4687,0.9249,0.346800,1.825,0.821900,1.0000,chr7:122319215:G:T,0.013680,NaN,NaN
2,CADPS2,AAC,assoc,allVariants,chr7:122319877:A:G_G,3,30,445,0,478,...,0.6128,0.9273,0.211300,1.403,0.720800,1.0000,chr7:122319877:A:G,0.037660,NaN,NaN
3,CADPS2,AAC,assoc,allVariants,chr7:122320660:G:A_A,0,14,461,3,478,...,0.5560,1.092,0.344300,2.144,0.798500,1.0000,chr7:122320660:G:A,0.014740,NaN,NaN
4,CADPS2,AAC,assoc,allVariants,chr7:122321156:T:C_C,0,11,465,2,478,...,0.5173,1.109,0.389100,2.378,0.790000,1.0000,chr7:122321156:T:C,0.011550,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13554,CADPS2,CAH,glm,allVariants,chr7:122886666:C:T_C,691,20,0,74,785,...,NaN,1.12946,0.373680,NaN,0.744588,1.0000,chr7:122886666:C:T,0.025055,971.0,0.325783
13555,CADPS2,CAH,assoc,allExonic,chr7:122621693:C:T_T,0,27,758,0,785,...,0.7125,1.522,NaN,3.253,0.274600,0.5492,chr7:122621693:C:T,0.017200,NaN,NaN
13556,CADPS2,CAH,assoc,allExonic,chr7:122698916:C:T_T,0,40,745,0,785,...,0.7056,1.268,NaN,2.279,0.426300,0.8525,chr7:122698916:C:T,0.025480,NaN,NaN
13557,CADPS2,CAH,glm,allExonic,chr7:122621693:C:T_C,757,26,0,2,785,...,NaN,1.28672,0.457647,NaN,0.581736,1.0000,chr7:122621693:C:T,0.016971,971.0,0.550851


In [70]:
summary_stats.columns

Index(['Gene', 'Ancestry', 'Analysis', 'Variant type', 'Variant', 'Hom Cases',
       'Het Cases', 'Hom Ref Cases', 'Missing Cases', 'Total Cases',
       'Carrier Freq in Cases', 'Hom Controls', 'Het Controls',
       'Hom Ref Controls', 'Missing Controls', 'Total Controls',
       'Carrier Freq in Controls', 'SNP', 'A1', 'F_A', 'F_U', 'A2', 'L95',
       'OR', 'SE', 'U95', 'P', 'BONF', 'ID', 'A1_FREQ', 'OBS_CT', 'Z_STAT'],
      dtype='object')

In [71]:
common_cols = ['Gene','Ancestry','Analysis','Variant type',	
               'ID', 'A1', 'A1_FREQ', 'P', 'BONF', 
               'OR', 'SE', 'L95', 'U95', 
               'Hom Cases', 'Het Cases', 'Hom Ref Cases', 
               'Missing Cases', 'Total Cases', 'Carrier Freq in Cases', 
               'Hom Controls', 'Het Controls', 'Hom Ref Controls', 
               'Missing Controls', 'Total Controls', 'Carrier Freq in Controls']

In [72]:
summary_stats = summary_stats[common_cols]
summary_stats

,Gene,Ancestry,Analysis,Variant type,ID,A1,A1_FREQ,P,BONF,OR,...,Hom Ref Cases,Missing Cases,Total Cases,Carrier Freq in Cases,Hom Controls,Het Controls,Hom Ref Controls,Missing Controls,Total Controls,Carrier Freq in Controls
0,CADPS2,AAC,assoc,allVariants,chr7:122318683:A:C,C,0.037660,0.720800,1.0000,0.9273,...,445,0,478,0.037657,2,62,751,0,815,0.040491
1,CADPS2,AAC,assoc,allVariants,chr7:122319215:G:T,T,0.013680,0.821900,1.0000,0.9249,...,462,3,478,0.013684,1,22,789,3,815,0.014778
2,CADPS2,AAC,assoc,allVariants,chr7:122319877:A:G,G,0.037660,0.720800,1.0000,0.9273,...,445,0,478,0.037657,2,62,751,0,815,0.040491
3,CADPS2,AAC,assoc,allVariants,chr7:122320660:G:A,A,0.014740,0.798500,1.0000,1.092,...,461,3,478,0.014737,1,20,793,1,815,0.013514
4,CADPS2,AAC,assoc,allVariants,chr7:122321156:T:C,C,0.011550,0.790000,1.0000,1.109,...,465,2,478,0.011555,0,17,798,0,815,0.010429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13554,CADPS2,CAH,glm,allVariants,chr7:122886666:C:T,T,0.025055,0.744588,1.0000,1.12946,...,0,74,785,0.985935,334,6,0,56,396,0.991176
13555,CADPS2,CAH,assoc,allExonic,chr7:122621693:C:T,T,0.017200,0.274600,0.5492,1.522,...,758,0,785,0.017197,0,9,387,0,396,0.011364
13556,CADPS2,CAH,assoc,allExonic,chr7:122698916:C:T,T,0.025480,0.426300,0.8525,1.268,...,745,0,785,0.025478,0,16,380,0,396,0.020202
13557,CADPS2,CAH,glm,allExonic,chr7:122621693:C:T,T,0.016971,0.581736,1.0000,1.28672,...,0,2,785,0.983397,385,9,0,2,396,0.988579


In [73]:
summary_stats['Variant type'].value_counts().to_frame()

,count
Variant type,
allVariants,13531
allExonic,28


In [74]:
summary_stats.to_csv(f"{RESU_DIR}/{analysis}/{GENE_NAME}_{analysis}_assocglm_SummaryStats.csv",  index=False)

### Significant stats

In [75]:
significant_stats = summary_stats[
    (summary_stats['P'] <= 0.05) &
    (summary_stats['Hom Cases'] != 0) &
    (summary_stats['Het Cases'] != 0) &
    (summary_stats['Hom Controls'] != 0) &
    (summary_stats['Het Controls'] != 0)
]
significant_stats

,Gene,Ancestry,Analysis,Variant type,ID,A1,A1_FREQ,P,BONF,OR,...,Hom Ref Cases,Missing Cases,Total Cases,Carrier Freq in Cases,Hom Controls,Het Controls,Hom Ref Controls,Missing Controls,Total Controls,Carrier Freq in Controls
712,CADPS2,AAC,assoc,allVariants,chr7:122636614:C:T,T,0.042020,0.017990,1.0,1.698,...,437,2,478,0.042017,1,39,774,1,815,0.025184
725,CADPS2,AAC,assoc,allVariants,chr7:122642574:T:C,C,0.042020,0.013630,1.0,1.741,...,437,2,478,0.042017,1,38,775,1,815,0.024570
1081,CADPS2,AAC,assoc,allVariants,chr7:122830346:T:A,A,0.062760,0.003228,1.0,1.722,...,423,0,478,0.062762,1,59,755,0,815,0.037423
1082,CADPS2,AAC,assoc,allVariants,chr7:122830348:C:A,A,0.062760,0.003228,1.0,1.722,...,423,0,478,0.062762,1,59,755,0,815,0.037423
1083,CADPS2,AAC,assoc,allVariants,chr7:122830463:C:T,T,0.062760,0.003228,1.0,1.722,...,423,0,478,0.062762,1,59,755,0,815,0.037423
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13306,CADPS2,CAH,glm,allVariants,chr7:122736871:T:C,C,0.016378,0.012005,1.0,0.350223,...,0,66,785,0.991655,338,8,0,50,396,0.988439
13330,CADPS2,CAH,glm,allVariants,chr7:122751933:C:G,G,0.010671,0.024358,1.0,5.50309,...,0,70,785,0.993007,349,2,0,45,396,0.997151
13518,CADPS2,CAH,glm,allVariants,chr7:122863970:C:T,T,0.010892,0.046510,1.0,3.60987,...,0,60,785,0.993103,349,2,0,45,396,0.997151
13533,CADPS2,CAH,glm,allVariants,chr7:122874108:T:C,C,0.010958,0.047693,1.0,3.53735,...,0,62,785,0.993084,348,2,0,46,396,0.997143


In [76]:
gene = 'CADPS2'
significant_stats.to_csv(f"{RESU_DIR}/{analysis}/{gene}_{analysis}_assocglm_SignificantVariantsNonAdjusted.csv", index=False)

In [77]:
significant_stats_bonf = significant_stats[(significant_stats['BONF'] <= 0.05)]
significant_stats_bonf

,Gene,Ancestry,Analysis,Variant type,ID,A1,A1_FREQ,P,BONF,OR,...,Hom Ref Cases,Missing Cases,Total Cases,Carrier Freq in Cases,Hom Controls,Het Controls,Hom Ref Controls,Missing Controls,Total Controls,Carrier Freq in Controls
5600,CADPS2,AMR,assoc,allVariants,chr7:122327354:C:T,T,0.02264,5.745000e-08,2.430000e-05,0.4803,...,1985,46,2122,0.022640,4,121,1277,84,1486,0.046006
5604,CADPS2,AMR,assoc,allVariants,chr7:122332314:G:A,A,0.01806,1.034000e-07,4.373000e-05,0.4538,...,2002,46,2122,0.018064,4,101,1294,87,1486,0.038956
5619,CADPS2,AMR,assoc,allVariants,chr7:122379319:T:C,C,0.02876,5.881000e-05,2.488000e-02,0.5989,...,1953,53,2122,0.028758,3,126,1272,85,1486,0.047109
5636,CADPS2,AMR,assoc,allVariants,chr7:122412185:C:G,G,0.04055,2.638000e-07,1.116000e-04,2.202,...,1929,26,2122,0.040553,1,53,1406,26,1486,0.018836
5710,CADPS2,AMR,assoc,allVariants,chr7:122457940:A:G,G,0.02703,2.747000e-08,1.162000e-05,0.5009,...,1981,32,2122,0.027033,1,150,1295,40,1486,0.052559
5730,CADPS2,AMR,assoc,allVariants,chr7:122468037:T:C,C,0.03887,1.234000e-06,5.220000e-04,2.115,...,1937,25,2122,0.038865,1,53,1412,20,1486,0.018759
5733,CADPS2,AMR,assoc,allVariants,chr7:122469116:A:G,G,0.05257,7.105000e-05,3.006000e-02,1.627,...,1904,1,2122,0.052570,2,94,1390,0,1486,0.032974
5785,CADPS2,AMR,assoc,allVariants,chr7:122539483:G:A,A,0.05631,5.209000e-05,2.203000e-02,1.613,...,1887,0,2122,0.056315,3,100,1383,0,1486,0.035666
5794,CADPS2,AMR,assoc,allVariants,chr7:122554232:G:T,T,0.04080,1.045000e-04,4.419000e-02,1.731,...,1949,2,2122,0.040802,2,67,1411,6,1486,0.023986
5795,CADPS2,AMR,assoc,allVariants,chr7:122554867:AT:A,A,0.04080,1.045000e-04,4.419000e-02,1.731,...,1949,2,2122,0.040802,2,67,1411,6,1486,0.023986


In [78]:
significant_stats_bonf.to_csv(f"{RESU_DIR}/{analysis}/CADPS2_{analysis}_assocglm_SignificantVariantsAdjusted.csv", index=False)

## Annotate significant variants

In [79]:
significant_stats[['chr', 'pos', 'ref', 'alt']] = significant_stats['ID'].str.split(':', expand=True)

# Prepare avinput DataFrame
avinput = significant_stats[['chr', 'pos', 'pos', 'ref', 'alt']]

# Save to tab-delimited file
avinput.to_csv(f'{RESU_DIR}/significant_variants.avinput', sep='\t', index=False, header=False)

/tmp/ipykernel_658/718702108.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  significant_stats[['chr', 'pos', 'ref', 'alt']] = significant_stats['ID'].str.split(':', expand=True)
/tmp/ipykernel_658/718702108.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  significant_stats[['chr', 'pos', 'ref', 'alt']] = significant_stats['ID'].str.split(':', expand=True)
/tmp/ipykernel_658/718702108.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_

In [80]:
BUILD = "hg38"
annotateClinvar = [
        "perl", f"{TOOL_DIR}/annovar/table_annovar.pl",
        f'{RESU_DIR}/significant_variants.avinput',
        f"{TOOL_DIR}/annovar/humandb/",
        "-buildver", f"{BUILD}",
        "-out", f'{RESU_DIR}/{analysis}/significant_variants',
        "-remove", 
        "-protocol", "clinvar_20250721",
        "-operation", "f",
        "--nopolish",
        "-nastring", ".",
    ]

subprocess.run(annotateClinvar, check=True)

-----------------------------------------------------------------
NOTICE: Processing operation=f protocol=clinvar_20250721
NOTICE: Finished reading 13 column headers for '-dbtype clinvar_20250721'

NOTICE: Running system command <annotate_variation.pl -filter -dbtype clinvar_20250721 -buildver hg38 -outfile /home/jupyter/workspace/ws_files/NahuTestR11/Results/NBA/significant_variants /home/jupyter/workspace/ws_files/NahuTestR11/Results/significant_variants.avinput /home/jupyter/tools/annovar/humandb/ -otherinfo>
NOTICE: the --dbtype clinvar_20250721 is assumed to be in generic ANNOVAR database format
NOTICE: Output file with variants matching filtering criteria is written to /home/jupyter/workspace/ws_files/NahuTestR11/Results/NBA/significant_variants.hg38_clinvar_20250721_dropped, and output file with other variants is written to /home/jupyter/workspace/ws_files/NahuTestR11/Results/NBA/significant_variants.hg38_clinvar_20250721_filtered
NOTICE: Processing next batch with 467 unique va

CompletedProcess(args=['perl', '/home/jupyter/tools/annovar/table_annovar.pl', '/home/jupyter/workspace/ws_files/NahuTestR11/Results/significant_variants.avinput', '/home/jupyter/tools/annovar/humandb/', '-buildver', 'hg38', '-out', '/home/jupyter/workspace/ws_files/NahuTestR11/Results/NBA/significant_variants', '-remove', '-protocol', 'clinvar_20250721', '-operation', 'f', '--nopolish', '-nastring', '.'], returncode=0)

In [81]:
multianno = pd.read_csv(f'{RESU_DIR}/{analysis}/significant_variants.hg38_multianno.txt', sep='\t', header=0)
multianno['ID'] = multianno['Chr'] + ":" + multianno['Start'].astype(str) + ":" + multianno['Ref'] + ":" + multianno['Alt']

annotated_significant = pd.merge(significant_stats, multianno, how='inner', on='ID')
annotated_significant

,Gene,Ancestry,Analysis,Variant type,ID,A1,A1_FREQ,P,BONF,OR,...,CLNREVSTAT,CLNSIG,ONCDN,ONCDISDB,ONCREVSTAT,ONC,SCIDN,SCIDISDB,SCIREVSTAT,SCI
0,CADPS2,AAC,assoc,allVariants,chr7:122636614:C:T,T,0.042020,0.017990,1.0,1.698,...,.,.,.,.,.,.,.,.,.,.
1,CADPS2,AAC,assoc,allVariants,chr7:122636614:C:T,T,0.042020,0.017990,1.0,1.698,...,.,.,.,.,.,.,.,.,.,.
2,CADPS2,AAC,assoc,allVariants,chr7:122642574:T:C,C,0.042020,0.013630,1.0,1.741,...,.,.,.,.,.,.,.,.,.,.
3,CADPS2,AAC,assoc,allVariants,chr7:122642574:T:C,C,0.042020,0.013630,1.0,1.741,...,.,.,.,.,.,.,.,.,.,.
4,CADPS2,AAC,assoc,allVariants,chr7:122830346:T:A,A,0.062760,0.003228,1.0,1.722,...,.,.,.,.,.,.,.,.,.,.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
903,CADPS2,CAH,glm,allVariants,chr7:122736871:T:C,C,0.016378,0.012005,1.0,0.350223,...,.,.,.,.,.,.,.,.,.,.
904,CADPS2,CAH,glm,allVariants,chr7:122751933:C:G,G,0.010671,0.024358,1.0,5.50309,...,.,.,.,.,.,.,.,.,.,.
905,CADPS2,CAH,glm,allVariants,chr7:122863970:C:T,T,0.010892,0.046510,1.0,3.60987,...,.,.,.,.,.,.,.,.,.,.
906,CADPS2,CAH,glm,allVariants,chr7:122874108:T:C,C,0.010958,0.047693,1.0,3.53735,...,.,.,.,.,.,.,.,.,.,.


In [82]:
significant_stats_bonf.to_csv(f"{RESU_DIR}/{analysis}/CADPS2_{analysis}_assocglm_SignificantVariantsAnnotated.csv", index=False)

## To-Do list

- Make all analyses into one single function, and call it with different assoc models, ANCESTRIES, variant groups in parallel

- Check exonic GLM +

- Add all outputs in single df by ANCESTRIES in first column, keep p<0.05